In [ ]:
# Cell 1 — Frozen single-seed final locked-TEST contract
import os, time, json, hashlib
from pathlib import Path
import numpy as np

# Change only RUN_SEED to select one of the five frozen publication seeds.
RUN_SEED=42
AUTHORIZE_LOCKED_TEST=True
LOCKED_TEST_ACK="I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST"
LOCKED_TEST_BUILD_ID="PANGEA_LOCKED_TEST_FIXED_20260913_A"

PUBLICATION_SEEDS=[42,1337,2026,31415,271828]
if RUN_SEED not in PUBLICATION_SEEDS:
    raise RuntimeError(f"Unexpected publication seed: {RUN_SEED}")

SHARED_FORWARD_PROTOCOL_SHA256="37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"
PARENT_GATED_PROTOCOL_SHA256="627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd"
PANGEA_FEATURE_BINDING_SHA256="c7052cc933ab70161088728ebf82d4944d60ba13f0d2a0d9548a3b7de453d2eb"
TASK_INSTRUCTION="Encode this meme for six independent annotations: offensive, vulgar, abuse, sarcastic, language (Bangla/English/Banglish), and zero or more of seven real target groups. Do not impose logical implications between tasks. "
TASK_INSTRUCTION_SHA=hashlib.sha256(TASK_INSTRUCTION.encode("utf-8")).hexdigest()
if TASK_INSTRUCTION_SHA!="bfef1e522a9433e1ca0c57a3a1bea49cd890d8428308b7ffdf6b24e4fe51667f":
    raise RuntimeError("Task instruction drift")

MODEL_ID="neulab/Pangea-7B-hf"
MODEL_REVISION="9394c5693ad814cea885d21c34c98b71fcd1fc1b"

SEED_BINDINGS={
    42:{
        "selected_gated_step":800,
        "validation_freeze_id":"059dac5a0b4362860ee5f6c6ec66200a34eb6c50f0a8d35b555c519921efa496",
        "parent_fingerprint":"be7c6b9aacac4b03c59bfe81a2d2438f9d6080b0ca63f6c46ff57cab394084e9",
        "selected_facetgate_step":932,
    },
    1337:{
        "selected_gated_step":800,
        "validation_freeze_id":"d2b524e59acaf17294ed761e4e0a7022044a3733067aa6e186b26ffd4a0ff48b",
        "parent_fingerprint":"ef8471b99c7107fe0f77ccd371b293eb23082cd3f466d9ec23abd5d45e885779",
        "selected_facetgate_step":932,
    },
    2026:{
        "selected_gated_step":932,
        "validation_freeze_id":"bb4495cdc6cf90647f8ebc42a5bd7e9ca2fa3a096199d2391f918acc5755ed42",
        "parent_fingerprint":"2eadd7122737eac9c5af8d14a2f7fbba166814ac896428f8ae2ab378b0a07e91",
        "selected_facetgate_step":932,
    },
    31415:{
        "selected_gated_step":932,
        "validation_freeze_id":"6e832d1f383eb85ac447590c0940f2b97dcc10667972888947c6e8f0fcf600f4",
        "parent_fingerprint":"4861eac7f6c382cc8d4818d8bea10b73ae2a7abb637f78ec0ecbd564c9cc6ccd",
        "selected_facetgate_step":600,
    },
    271828:{
        "selected_gated_step":932,
        "validation_freeze_id":"6d9401b7e73d3379eb5972c8ce2e6bb83819ecca498ff0247db7a36b4c789cb9",
        "parent_fingerprint":"88e24c6a06f3a70ed9f7f39f1f87603321141a7169ea215d04f4ec11d3fcad9b",
        "selected_facetgate_step":600,
    },
}
_BINDING=SEED_BINDINGS[RUN_SEED]
MODEL_CFG={
    "display_name":"Pangea-7B-HF",
    "artifact_tag":"Pangea7BHF",
    "selected_gated_step":int(_BINDING["selected_gated_step"]),
    "validation_freeze_id":str(_BINDING["validation_freeze_id"]),
    "baseline_reproduction_tolerance":0.002,
    "expected_hidden":3584,
}
EXPECTED_PARENT_FINGERPRINT=str(_BINDING["parent_fingerprint"])
EXPECTED_FACETGATE_STEP=int(_BINDING["selected_facetgate_step"])

BINARY_TASKS=["offensive","vulgar","abuse","sarcastic"]
LANGUAGE_LABELS=["Bangla","English","Banglish"]
TARGET_LABELS=["Individual","Gender","Political","Social","National","Religious","Others"]

DATASET_RELEASE="M3Offend"
LABEL_SHA="9cb9afdd88c9a750d3eb6f36aa512dc6ad5b3067aedfac79b138b7fef64a9bb6"
OCR_SHA="eed35bf77f749e98d9222fd571d149b99c4c6a610e29bc5db5783d3489492c0c"
SPLIT_SHA="bbcf7e13fc0aab4ac3b5f58a2ebf8880740ed60ad7ecb2cb61910b0497d1b4f0"
CANONICAL_ROW_SHA="440f8e6982cf4272fc0b6c3e3e6fc9b079a02ce01cd5350a02be0d4a9fa1cf39"
TEST_IDENTITY_SHA="5f6ac881e9026d682f2f41d65ed1f08e1861d8c8f030bae04d98f9b9db6419db"
IMAGE_BUNDLE_SHA="9e134f26c63c23ba871c954ee1d7e03499a47ac3f9bd49d12e0c5ffa977665af"
EXPECTED_SUPPORTS={
    "offensive":{"No":694,"Yes":238},
    "vulgar":{"No":695,"Yes":237},
    "abuse":{"No":813,"Yes":119},
    "sarcastic":{"No":259,"Yes":673},
    "language":{"Bangla":605,"English":181,"Banglish":146},
    "target":{"Individual":147,"Gender":147,"Political":114,"Social":108,"National":34,"Religious":19,"Others":89},
}

GATED_SOURCE_OVERRIDE=""
GATED_THRESHOLDS_OVERRIDE=""
INPUT_ROOT=Path("/kaggle/input")
OUTPUT_ROOT=Path(f"/kaggle/working/M3Offend_9318_Pangea7BHF_FacetGate_Seed{RUN_SEED}_FINAL_LOCKED_TEST")
FEATURE_DIR=OUTPUT_ROOT/"locked_test_features"
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
FEATURE_DIR.mkdir(parents=True,exist_ok=True)

SESSION_START_UNIX=time.time()
TEST_SOFT_STOP_SECONDS=11*3600

print("[SINGLE-SEED LOCKED-TEST CONTRACT]",json.dumps({
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "seed":RUN_SEED,
    "parent_gated_selected_step":MODEL_CFG["selected_gated_step"],
    "facetgate_selected_step":EXPECTED_FACETGATE_STEP,
    "parent_fingerprint":EXPECTED_PARENT_FINGERPRINT,
    "protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256,
    "ordered_test_identity_sha256":TEST_IDENTITY_SHA,
    "authorization":AUTHORIZE_LOCKED_TEST,
    "build_id":LOCKED_TEST_BUILD_ID,
},indent=2))


In [ ]:

# CELL 2 — PINNED KAGGLE RUNTIME (NO SCIENTIFIC CHANGES)
import importlib.metadata as md, subprocess, sys, os, json
protected=["torch","torchvision","numpy","pandas","scikit-learn","scipy","Pillow"]
before={}
for pkg in protected:
    try: before[pkg]=md.version(pkg)
    except Exception: pass
constraints=Path("/kaggle/working/facetgate_constraints.txt")
constraints.write_text("\n".join(f"{k}=={v}" for k,v in before.items())+"\n",encoding="utf-8")
cmd=[sys.executable,"-m","pip","install","--no-cache-dir","--upgrade-strategy","only-if-needed",
     "-c",str(constraints),
     "transformers==5.15.0","peft==0.20.0","accelerate==1.14.0",
     "bitsandbytes==0.50.1","safetensors==0.8.0"]
print("[PIP]"," ".join(cmd),flush=True)
subprocess.run(cmd,check=True)
after={}
for pkg in protected:
    try: after[pkg]=md.version(pkg)
    except Exception: pass
if before!=after:
    raise RuntimeError(f"Protected base package drift: before={before} after={after}")
print("[RUNTIME PASS]",after,flush=True)


In [ ]:

# CELL 3 — ZERO-COMPUTE PARENT GATED CHECKPOINT + THRESHOLD PREFLIGHT
import os, json, re, hashlib, zipfile, shutil, tempfile
from pathlib import Path
import torch

INPUT_ROOT=Path("/kaggle/input")
RESOLVE_ROOT=Path("/kaggle/working/facetgate_parent_resolution")
RESOLVE_ROOT.mkdir(parents=True,exist_ok=True)

def sha256_file(p,chunk=8<<20):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""): h.update(b)
    return h.hexdigest()

def unwrap_state(obj):
    if isinstance(obj,dict) and isinstance(obj.get("discriminative"),dict): obj=obj["discriminative"]
    elif isinstance(obj,dict) and isinstance(obj.get("state_dict"),dict): obj=obj["state_dict"]
    return obj

def flat_tensor_keys(obj):
    obj=unwrap_state(obj)
    if not isinstance(obj,dict): return []
    return [str(k) for k,v in obj.items() if torch.is_tensor(v)]

def expected_pangea_gated_discriminative_keys():
    keys={"missing_vision","missing_text"}
    for stem in ("image_projection","text_projection"):
        keys.update({f"{stem}.0.weight",f"{stem}.0.bias",f"{stem}.1.weight",f"{stem}.1.bias"})
    keys.update({"gate.weight","gate.bias","fusion_norm.weight","fusion_norm.bias","shared.0.weight","shared.0.bias"})
    for task in BINARY_TASKS+["language","target"]:
        keys.update({
            f"adapters.{task}.norm.weight",f"adapters.{task}.norm.bias",
            f"adapters.{task}.net.0.weight",f"adapters.{task}.net.0.bias",
            f"adapters.{task}.net.3.weight",f"adapters.{task}.net.3.bias",
        })
    for task in BINARY_TASKS:
        keys.update({f"binary_heads.{task}.weight",f"binary_heads.{task}.bias"})
    keys.update({"language_head.weight","language_head.bias","target_head.weight","target_head.bias"})
    return keys

EXPECTED_PANGEA_GATED_DISCRIMINATIVE_KEYS=expected_pangea_gated_discriminative_keys()

def compatible_heads(path):
    try: obj=torch.load(path,map_location="cpu",weights_only=True)
    except Exception: return False
    obj=unwrap_state(obj)
    if not isinstance(obj,dict) or not all(torch.is_tensor(v) for v in obj.values()): return False
    observed=set(map(str,obj.keys()))
    # Exact Frozen Common Gated v1.1 Pangea classifier layout.  This intentionally
    # rejects a semantically similar but state-incompatible adapter implementation.
    if observed!=EXPECTED_PANGEA_GATED_DISCRIMINATIVE_KEYS: return False
    if any(".block." in k for k in observed): return False
    if not all(".net." in k for k in observed if k.startswith("adapters.") and (".0." in k or ".3." in k)): return False
    return True

def checkpoint_dir_ok(d):
    d=Path(d)
    return ((d/"classifier_heads.pt").is_file() and
            (d/"adapter"/"adapter_config.json").is_file() and
            any((d/"adapter"/x).is_file() for x in ["adapter_model.safetensors","adapter_model.bin"]) and
            compatible_heads(d/"classifier_heads.pt"))

def _collect_scalar_seed_values(obj):
    vals=[]
    if isinstance(obj,dict):
        for k,v in obj.items():
            key=str(k).strip().casefold()
            if key in {"seed","run_seed","training_seed","model_seed"} and isinstance(v,(int,float)) and float(v).is_integer():
                vals.append(int(v))
            if isinstance(v,(dict,list,tuple)):
                vals.extend(_collect_scalar_seed_values(v))
    elif isinstance(obj,(list,tuple)):
        for v in obj:
            if isinstance(v,(dict,list,tuple)):
                vals.extend(_collect_scalar_seed_values(v))
    return vals

def checkpoint_seed_provenance(d):
    """Require exact scalar seed provenance from the matched Gated checkpoint manifest."""
    d=Path(d)
    mf=d/"checkpoint_manifest.json"
    if not mf.is_file():
        return False,{"reason":"checkpoint_manifest.json missing","seeds":[]}
    try:
        obj=json.loads(mf.read_text(encoding="utf-8"))
    except Exception as e:
        return False,{"reason":f"checkpoint_manifest unreadable: {type(e).__name__}","seeds":[]}
    # Exact parent identity checks before any backbone forward.
    if obj.get("model_id") not in (None, MODEL_ID):
        return False,{"reason":f"model_id mismatch: {obj.get('model_id')}","seeds":[]}
    if obj.get("model_revision") not in (None, MODEL_REVISION):
        return False,{"reason":f"model_revision mismatch: {obj.get('model_revision')}","seeds":[]}
    if obj.get("frozen_common_gated_protocol_sha256") not in (None, PARENT_GATED_PROTOCOL_SHA256):
        return False,{"reason":"Frozen Common Gated protocol SHA mismatch","seeds":[]}
    for _key in ("test_forward_count","test_model_forwards","test_forwards","development_test_forwards"):
        if _key in obj:
            try:
                if int(obj[_key]) != 0:
                    return False,{"reason":f"{_key} is nonzero: {obj[_key]}","seeds":[]}
            except Exception:
                return False,{"reason":f"{_key} is not an integer zero","seeds":[]}
    vals=sorted(set(_collect_scalar_seed_values(obj)))
    if not vals:
        return False,{"reason":"no scalar seed provenance in checkpoint_manifest.json","seeds":[]}
    ok=(vals==[int(RUN_SEED)])
    return ok,{"reason":"ok" if ok else "seed mismatch/conflict","seeds":vals}

def extract_checkpoint_from_zip(zp,step):
    zp=Path(zp)
    with zipfile.ZipFile(zp) as z:
        names=z.namelist()
        patterns=[f"checkpoint-{step:04d}/classifier_heads.pt",f"checkpoint-{step}/classifier_heads.pt"]
        heads=[n for n in names if any(n.endswith(p) for p in patterns)]
        if len(heads)!=1: return None
        prefix=heads[0][:-len("classifier_heads.pt")]
        needed=[n for n in names if n.startswith(prefix)]
        if not any(n.endswith("adapter/adapter_config.json") for n in needed): return None
        tag=hashlib.sha256((str(zp.resolve())+"|"+prefix).encode()).hexdigest()[:16]
        out=RESOLVE_ROOT/f"zip_{tag}"
        if out.exists(): shutil.rmtree(out)
        out.mkdir(parents=True)
        for n in needed:
            rel=n[len(prefix):]
            if not rel or n.endswith("/"): continue
            dest=out/rel; dest.parent.mkdir(parents=True,exist_ok=True)
            with z.open(n) as fi, open(dest,"wb") as fo: shutil.copyfileobj(fi,fo)
        if checkpoint_dir_ok(out): return out
    return None

def discover_parent_candidates():
    selected=int(MODEL_CFG["selected_gated_step"])
    cands=[]
    if GATED_SOURCE_OVERRIDE:
        p=Path(GATED_SOURCE_OVERRIDE)
        if not p.exists(): raise FileNotFoundError(f"GATED_SOURCE_OVERRIDE does not exist: {p}")
        if p.is_dir():
            if checkpoint_dir_ok(p): cands.append(p)
            else:
                for name in [f"checkpoint-{selected:04d}",f"checkpoint-{selected}"]:
                    for q in p.rglob(name):
                        if q.is_dir() and checkpoint_dir_ok(q): cands.append(q)
        elif p.is_file() and zipfile.is_zipfile(p):
            q=extract_checkpoint_from_zip(p,selected)
            if q is not None: cands.append(q)
    else:
        for name in [f"checkpoint-{selected:04d}",f"checkpoint-{selected}"]:
            for q in INPUT_ROOT.rglob(name):
                try:
                    if q.is_dir() and checkpoint_dir_ok(q): cands.append(q)
                except Exception: pass
        # ZIP fallback supports the Pangea frozen-output safeguard and portable Gated outputs.
        for zp in INPUT_ROOT.rglob("*.zip"):
            try:
                if zp.stat().st_size>8_000_000_000: continue
                q=extract_checkpoint_from_zip(zp,selected)
                if q is not None: cands.append(q)
            except Exception:
                pass
    # Hard same-seed provenance filter before expensive model loading / feature extraction.
    seeded=[]
    seed_rejections=[]
    for d in cands:
        ok,ev=checkpoint_seed_provenance(d)
        if ok:
            seeded.append(d)
        else:
            seed_rejections.append({"path":str(d),**ev})
    cands=seeded
    if not cands:
        raise RuntimeError(
            f"No compatible Pangea Gated checkpoint with machine-readable seed={RUN_SEED} provenance "
            f"was found at required step {selected}. Rejections={seed_rejections}. "
            "Attach the completed Gated output for the SAME RUN_SEED."
        )

    # Content dedup: head bytes + adapter bytes/config define the exact frozen parent state.
    unique={}
    for d in cands:
        am=next((d/"adapter"/x for x in ["adapter_model.safetensors","adapter_model.bin"] if (d/"adapter"/x).is_file()),None)
        if am is None: continue
        fp=hashlib.sha256(
            (sha256_file(d/"classifier_heads.pt")+"|"+sha256_file(am)+"|"+sha256_file(d/"adapter"/"adapter_config.json")).encode()
        ).hexdigest()
        unique.setdefault(fp,[]).append(d)
    if not unique:
        raise RuntimeError(
            f"No compatible matched Gated same-seed checkpoint step {selected} found. "
            "Attach the COMPLETE same-seed Gated output for this backbone, including adapter/ and classifier_heads.pt."
        )
    if len(unique)>1:
        report={k:[str(x) for x in v] for k,v in unique.items()}
        raise RuntimeError(
            "Multiple different compatible Gated checkpoint states were found. "
            f"Set GATED_SOURCE_OVERRIDE to the intended completed same-seed Gated output. candidates={report}"
        )
    fp,paths=next(iter(unique.items()))
    chosen=sorted(paths,key=lambda p:(len(str(p)),str(p)))[0]
    return chosen,fp

PARENT_CHECKPOINT_DIR,PARENT_CHECKPOINT_FINGERPRINT=discover_parent_candidates()
PARENT_SEED_PROVENANCE_OK,PARENT_SEED_PROVENANCE=checkpoint_seed_provenance(PARENT_CHECKPOINT_DIR)
if not PARENT_SEED_PROVENANCE_OK: raise RuntimeError(f"Parent seed provenance failed: {PARENT_SEED_PROVENANCE}")
PARENT_HEADS=PARENT_CHECKPOINT_DIR/"classifier_heads.pt"
PARENT_ADAPTER=PARENT_CHECKPOINT_DIR/"adapter"
print("[PARENT GATED CHECKPOINT PASS]",{
    "path":str(PARENT_CHECKPOINT_DIR),
    "selected_step":MODEL_CFG["selected_gated_step"],
    "fingerprint":PARENT_CHECKPOINT_FINGERPRINT,
    "heads_sha256":sha256_file(PARENT_HEADS),
    "seed_provenance":PARENT_SEED_PROVENANCE,
},flush=True)

# Frozen Gated threshold discovery.
def normalized_thresholds(obj):
    if isinstance(obj,dict) and isinstance(obj.get("thresholds"),dict): obj=obj["thresholds"]
    if not isinstance(obj,dict): return None
    binary={}
    for t in BINARY_TASKS:
        keys=[t,t.lower(),f"threshold_{t}",f"{t}_threshold"]
        hit=next((obj[k] for k in keys if k in obj and isinstance(obj[k],(int,float))),None)
        if hit is None and isinstance(obj.get("binary"),dict):
            hit=next((obj["binary"][k] for k in keys if k in obj["binary"] and isinstance(obj["binary"][k],(int,float))),None)
        if hit is None: return None
        binary[t]=float(hit)
    target={}
    nested=obj.get("target") if isinstance(obj.get("target"),dict) else {}
    low={str(k).casefold():v for k,v in obj.items()}
    nlow={str(k).casefold():v for k,v in nested.items()}
    for lab in TARGET_LABELS:
        candidates=[lab,lab.casefold(),f"target::{lab}",f"target_{lab}",f"target.{lab}"]
        val=None
        for k in candidates:
            if k in obj and isinstance(obj[k],(int,float)): val=obj[k]; break
            if k.casefold() in low and isinstance(low[k.casefold()],(int,float)): val=low[k.casefold()]; break
            if k in nested and isinstance(nested[k],(int,float)): val=nested[k]; break
            if k.casefold() in nlow and isinstance(nlow[k.casefold()],(int,float)): val=nlow[k.casefold()]; break
        if val is None: return None
        target[lab]=float(val)
    vals=list(binary.values())+list(target.values())
    if not all(0.0<=x<=1.0 for x in vals): return None
    return {"binary":binary,"target":target}

def candidate_threshold_files():
    files=[]
    if GATED_THRESHOLDS_OVERRIDE:
        p=Path(GATED_THRESHOLDS_OVERRIDE)
        if not p.is_file(): raise FileNotFoundError(p)
        return [p]
    # Prefer the mounted tree containing the chosen checkpoint.
    roots=[]
    p=PARENT_CHECKPOINT_DIR
    for _ in range(5):
        roots.append(p); p=p.parent
    for root in roots:
        for name in ["thresholds.json","validation_thresholds.json"]:
            files += list(root.rglob(name))
    # Mounted-output fallback.
    for name in ["thresholds.json","validation_thresholds.json"]:
        files += list(INPUT_ROOT.rglob(name))
    return list(dict.fromkeys(Path(x) for x in files))

valid={}
for p in candidate_threshold_files():
    try:
        obj=json.loads(p.read_text(encoding="utf-8"))
        norm=normalized_thresholds(obj)
        if norm is None: continue
        payload=json.dumps(norm,sort_keys=True,separators=(",",":"))
        valid.setdefault(hashlib.sha256(payload.encode()).hexdigest(),[]).append((p,norm))
    except Exception:
        pass
if not valid:
    raise RuntimeError(
        "No complete frozen Gated validation threshold file was found. "
        "Attach the completed same-seed Gated output containing thresholds.json, or set GATED_THRESHOLDS_OVERRIDE."
    )
if len(valid)>1:
    # Prefer a threshold file inside the chosen parent's local ancestor tree if that makes identity unique.
    near={}
    parent_text=str(PARENT_CHECKPOINT_DIR)
    for k,vals in valid.items():
        if any(str(p).startswith(str(PARENT_CHECKPOINT_DIR.parent.parent)) for p,_ in vals):
            near[k]=vals
    if len(near)==1:
        valid=near
if len(valid)>1:
    raise RuntimeError(
        "Multiple different complete Gated threshold sets were found. "
        "Set GATED_THRESHOLDS_OVERRIDE to the exact same-seed threshold file."
    )
thr_sha,vals=next(iter(valid.items()))
THRESHOLD_SOURCE,THRESHOLDS_NORMALIZED=sorted(vals,key=lambda z:(len(str(z[0])),str(z[0])))[0]
(OUTPUT_ROOT/"parent_gated_thresholds.normalized.json").write_text(
    json.dumps(THRESHOLDS_NORMALIZED,indent=2),encoding="utf-8"
)
binding={
    "schema":"M3Offend_FacetGate_v1_0_BASE_GATED_BINDING",
    "protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256,
    "parent_gated_protocol_sha256":PARENT_GATED_PROTOCOL_SHA256,
    "model_id":MODEL_ID,"model_revision":MODEL_REVISION,"seed":RUN_SEED,
    "selected_gated_step":MODEL_CFG["selected_gated_step"],
    "checkpoint_dir":str(PARENT_CHECKPOINT_DIR),
    "checkpoint_fingerprint":PARENT_CHECKPOINT_FINGERPRINT,
    "checkpoint_seed_provenance":PARENT_SEED_PROVENANCE,
    "classifier_heads_sha256":sha256_file(PARENT_HEADS),
    "threshold_source":str(THRESHOLD_SOURCE),
    "normalized_threshold_sha256":thr_sha,
    "test_model_forwards":0,
}
(OUTPUT_ROOT/"BASE_GATED_BINDING.json").write_text(json.dumps(binding,indent=2),encoding="utf-8")
print("[PARENT GATED THRESHOLDS PASS]",{
    "source":str(THRESHOLD_SOURCE),"normalized_sha256":thr_sha,"TEST_forwards":0
},flush=True)



if PARENT_CHECKPOINT_FINGERPRINT != EXPECTED_PARENT_FINGERPRINT:
    raise RuntimeError(
        f"Frozen parent fingerprint mismatch: observed={PARENT_CHECKPOINT_FINGERPRINT} "
        f"expected={EXPECTED_PARENT_FINGERPRINT}"
    )

FACET_RESOLVE_ROOT=OUTPUT_ROOT/"facetgate_validation_state"
FACET_RESOLVE_ROOT.mkdir(parents=True,exist_ok=True)

def _json_read(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def _sha_json_normalized(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(",",":")).encode()).hexdigest()

def _facet_candidate_ok(root):
    root=Path(root)
    rp=root/"facetgate_result.json"
    cp=root/"selected_checkpoint.pt"
    tp=root/"validation_thresholds.json"
    bp=root/"BASE_GATED_BINDING.json"
    if not all(x.is_file() for x in [rp,cp,tp,bp]):
        return None
    try:
        result=_json_read(rp)
        if result.get("schema")!="M3Offend_FacetGate_v1_0_1_PublicationSeed_Result": return None
        if result.get("protocol_sha256")!=SHARED_FORWARD_PROTOCOL_SHA256: return None
        if result.get("model_id")!=MODEL_ID or result.get("model_revision")!=MODEL_REVISION: return None
        if int(result.get("seed",-1))!=RUN_SEED: return None
        if int(result.get("selected_facetgate_step",-1))!=EXPECTED_FACETGATE_STEP: return None
        if int(result.get("parent_gated_selected_step",-1))!=MODEL_CFG["selected_gated_step"]: return None
        if result.get("verdict")!="PASS": return None
        if int(result.get("locked_test_model_forwards",-1))!=0: return None

        thr=_json_read(tp)
        if thr.get("source_split")!="validation" or int(thr.get("source_rows",-1))!=932: return None
        tmap=thr.get("thresholds")
        if not isinstance(tmap,dict) or set(tmap)!=set(TARGET_LABELS): return None
        if not all(isinstance(tmap[k],(int,float)) and 0.0<=float(tmap[k])<=1.0 for k in TARGET_LABELS): return None

        bind=_json_read(bp)
        if bind.get("protocol_sha256")!=SHARED_FORWARD_PROTOCOL_SHA256: return None
        if bind.get("model_id")!=MODEL_ID or bind.get("model_revision")!=MODEL_REVISION: return None
        if int(bind.get("seed",-1))!=RUN_SEED: return None
        if int(bind.get("selected_gated_step",-1))!=MODEL_CFG["selected_gated_step"]: return None
        if bind.get("checkpoint_fingerprint")!=PARENT_CHECKPOINT_FINGERPRINT: return None
        if int(bind.get("test_model_forwards",-1))!=0: return None

        state=torch.load(cp,map_location="cpu",weights_only=False)
        if int(state.get("step",-1))!=EXPECTED_FACETGATE_STEP: return None
        fs=state.get("facetgate_state")
        if not isinstance(fs,dict) or not fs or not all(torch.is_tensor(v) for v in fs.values()): return None

        return {
            "root":root,
            "result":result,
            "thresholds":thr,
            "binding":bind,
            "checkpoint":cp,
            "checkpoint_sha256":sha256_file(cp),
            "threshold_sha256":_sha_json_normalized(thr),
            "result_sha256":sha256_file(rp),
        }
    except Exception:
        return None

facet_candidates=[]
for rp in INPUT_ROOT.rglob("facetgate_result.json"):
    cand=_facet_candidate_ok(rp.parent)
    if cand is not None:
        facet_candidates.append(cand)

# Also accept the development RESULTS.zip mounted as a Kaggle dataset input.
for zp in INPUT_ROOT.rglob("*.zip"):
    try:
        with zipfile.ZipFile(zp) as z:
            names=z.namelist()
            result_entries=[n for n in names if n.endswith("facetgate_result.json")]
            for rn in result_entries:
                prefix=rn[:-len("facetgate_result.json")]
                required=[
                    prefix+"facetgate_result.json",
                    prefix+"selected_checkpoint.pt",
                    prefix+"validation_thresholds.json",
                    prefix+"BASE_GATED_BINDING.json",
                ]
                if not all(n in names for n in required):
                    continue
                tag=hashlib.sha256((str(zp)+"|"+prefix).encode()).hexdigest()[:16]
                dest=FACET_RESOLVE_ROOT/f"zip_{tag}"
                if dest.exists(): shutil.rmtree(dest)
                dest.mkdir(parents=True)
                for name in required:
                    with z.open(name) as src, open(dest/Path(name).name,"wb") as dst:
                        shutil.copyfileobj(src,dst)
                cand=_facet_candidate_ok(dest)
                if cand is not None:
                    facet_candidates.append(cand)
    except Exception:
        pass

if not facet_candidates:
    raise RuntimeError(
        f"No exact completed FacetGate validation state found for seed={RUN_SEED}. "
        "Attach that seed's completed FacetGate output (directory or RESULTS.zip) plus the matching Gated parent."
    )

unique={}
for c in facet_candidates:
    key=(c["checkpoint_sha256"],c["threshold_sha256"],c["result_sha256"])
    unique.setdefault(key,[]).append(c)
if len(unique)>1:
    raise RuntimeError(
        "Multiple different frozen FacetGate validation states were found for this seed. "
        "Remove ambiguous inputs before opening locked TEST."
    )
FACET_STATE=sorted(next(iter(unique.values())),key=lambda x:(len(str(x["root"])),str(x["root"])))[0]
FACET_SELECTED_CHECKPOINT=Path(FACET_STATE["checkpoint"])
FACET_SELECTED_CHECKPOINT_SHA256=FACET_STATE["checkpoint_sha256"]
FACET_VALIDATION_THRESHOLDS=dict(FACET_STATE["thresholds"]["thresholds"])
FACET_VALIDATION_THRESHOLD_SHA256=FACET_STATE["threshold_sha256"]

print("[FROZEN FACETGATE STATE PASS]",json.dumps({
    "seed":RUN_SEED,
    "selected_facetgate_step":EXPECTED_FACETGATE_STEP,
    "checkpoint":str(FACET_SELECTED_CHECKPOINT),
    "checkpoint_sha256":FACET_SELECTED_CHECKPOINT_SHA256,
    "validation_thresholds":FACET_VALIDATION_THRESHOLDS,
    "validation_threshold_sha256":FACET_VALIDATION_THRESHOLD_SHA256,
    "TEST_model_forwards_before_open":0,
},indent=2))


In [ ]:
# CELL 4 — CURRENT 9,318 DATASET AUDIT + DETERMINISTIC VALIDATION PHASH GROUPING
import os, json, re, ast, hashlib
from pathlib import Path
import pandas as pd, numpy as np

INPUT_ROOT=Path("/kaggle/input")

def sha256_file(path,chunk=8<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def read_table(path):
    p=Path(path)
    if p.suffix.lower()==".csv": return pd.read_csv(p)
    if p.suffix.lower() in {".jsonl",".ndjson"}: return pd.read_json(p,lines=True)
    obj=json.loads(p.read_text(encoding="utf-8"))
    if isinstance(obj,list): return pd.DataFrame(obj)
    if isinstance(obj,dict):
        for k in ["data","records","items","annotations","samples"]:
            if isinstance(obj.get(k),list): return pd.DataFrame(obj[k])
        if obj and all(isinstance(v,dict) for v in obj.values()):
            return pd.DataFrame([{"__dict_key__":k,**v} for k,v in obj.items()])
    raise ValueError(f"Unsupported JSON structure: {p}")

def resolve_col(df,cands,what,required=True):
    low={str(c).casefold():c for c in df.columns}
    for c in cands:
        if c.casefold() in low: return low[c.casefold()]
    if required: raise KeyError(f"Cannot resolve {what}; columns={list(df.columns)}")
    return None

def image_name(v): return str(v).replace("\\","/").split("/")[-1]
def norm_key(v):
    s=image_name(v).strip().casefold()
    return re.sub(r"\.(jpg|jpeg|png|webp|bmp)$","",s)

def norm_binary(v,task):
    if isinstance(v,(bool,np.bool_)): return "Yes" if bool(v) else "No"
    s=str(v).strip().casefold()
    if s in {"yes","y","1","true","positive","pos"}: return "Yes"
    if s in {"no","n","0","false","negative","neg"}: return "No"
    raise ValueError(f"{task}: bad binary label {v!r}")

def norm_language(v):
    s=" ".join(str(v).strip().split()).casefold()
    aliases={
        "bangla":"Bangla","bengali":"Bangla","english":"English",
        "banglish":"Banglish","banglish (bangla + english)":"Banglish",
        "banglish(bangla + english)":"Banglish","bangla + english":"Banglish",
        "bangla+english":"Banglish","bengali + english":"Banglish",
        "romanized bangla":"Banglish","romanised bangla":"Banglish",
    }
    if s not in aliases: raise ValueError(f"bad language {v!r}")
    return aliases[s]

def norm_target(v):
    if v is None or (isinstance(v,float) and np.isnan(v)): vals=[]
    elif isinstance(v,(list,tuple,set)): vals=list(v)
    elif isinstance(v,str):
        s=v.strip()
        if not s: vals=[]
        else:
            try:
                q=json.loads(s); vals=q if isinstance(q,list) else [q]
            except Exception:
                try:
                    q=ast.literal_eval(s); vals=list(q) if isinstance(q,(list,tuple,set)) else [q]
                except Exception: vals=re.split(r"[,;|]",s)
    else: vals=[v]
    canon={x.casefold():x for x in TARGET_LABELS}; canon["other"]="Others"
    out=[]
    for x in vals:
        sx=str(x).strip()
        if not sx or sx.casefold() in {"none","no target","null","nan"}: continue
        if sx.casefold() not in canon: raise ValueError(f"bad target {x!r}")
        c=canon[sx.casefold()]
        if c not in out: out.append(c)
    return [x for x in TARGET_LABELS if x in out]

def parse_labelstudio(df):
    rows=[]; errors=[]
    for rn,r in df.iterrows():
        try:
            data=r.get("data",{})
            if isinstance(data,str): data=json.loads(data)
            image_value=None
            if isinstance(data,dict):
                for k in ["image","image_id","image_name","filename","file_name","source_image_name"]:
                    if data.get(k) not in [None,""]:
                        image_value=data[k]; break
            if image_value is None:
                for k in ["image","image_id","image_name","filename","file_name","source_image_name","id"]:
                    if k in r and pd.notna(r[k]): image_value=r[k]; break
            anns=r.get("annotations",[])
            if isinstance(anns,str): anns=json.loads(anns)
            if not isinstance(anns,list) or not anns: raise ValueError("no annotations")
            coll={k:[] for k in BINARY_TASKS+["language","target"]}
            for ann in anns:
                for res in (ann.get("result",[]) if isinstance(ann,dict) else []):
                    if not isinstance(res,dict): continue
                    frm=str(res.get("from_name","")).casefold()
                    val=res.get("value",{}) or {}
                    choices=val.get("choices",val.get("labels",[]))
                    if isinstance(choices,str): choices=[choices]
                    task=next((k for k in coll if k in frm),None)
                    if task and choices: coll[task].append(list(choices))
            item={"record_key":norm_key(image_value),"label_source_id":str(image_value)}
            for t in BINARY_TASKS:
                vals=[norm_binary(b[0],t) for b in coll[t] if len(b)==1]
                if not vals or len(set(vals))!=1: raise ValueError(f"{t} non-unanimous/missing {vals}")
                item[t]=vals[0]
            lv=[norm_language(b[0]) for b in coll["language"] if len(b)==1]
            if not lv or len(set(lv))!=1: raise ValueError(f"language non-unanimous/missing {lv}")
            item["language"]=lv[0]
            tv=[tuple(norm_target(b)) for b in coll["target"] if b]
            if not tv: tv=[tuple()]
            if len(set(tv))!=1: raise ValueError(f"target non-unanimous {tv}")
            item["target_list"]=list(tv[0])
            rows.append(item)
        except Exception as e:
            errors.append({"row":int(rn),"error":repr(e)})
    if errors:
        pd.DataFrame(errors).to_csv(OUTPUT_ROOT/"label_domain_errors.csv",index=False)
        raise RuntimeError(f"Label parse errors={len(errors)} first={errors[:3]}")
    return pd.DataFrame(rows)

def parse_labels(path):
    df=read_table(path).copy()
    if "annotations" in df.columns:
        out=parse_labelstudio(df)
    else:
        idc=resolve_col(df,["image_id","image_name","source_image_name","source_image","image","filename","file_name","path","name","id","__dict_key__"],"label id")
        out=pd.DataFrame({"record_key":df[idc].map(norm_key),"label_source_id":df[idc].astype(str)})
        for t in BINARY_TASKS:
            c=resolve_col(df,[t],t); out[t]=df[c].map(lambda x:norm_binary(x,t))
        lc=resolve_col(df,["language","lang"],"language"); out["language"]=df[lc].map(norm_language)
        tc=resolve_col(df,["target","targets"],"target"); out["target_list"]=df[tc].map(norm_target)
    if out.record_key.duplicated().any(): raise RuntimeError("duplicate label IDs")
    return out

def parse_ocr(path):
    df=read_table(path).copy()
    idc=resolve_col(df,["image_id","image_name","source_image_name","source_image","image","filename","file_name","path","name","id","__dict_key__"],"OCR id")
    tc=resolve_col(df,["cleaned_ocr","ocr_cleaned","ocr","text","ocr_text","clean_text"],"OCR text")
    out=pd.DataFrame({"record_key":df[idc].map(norm_key),"cleaned_ocr":df[tc].fillna("").astype(str)})
    if out.record_key.duplicated().any(): raise RuntimeError("duplicate OCR IDs")
    return out

def parse_manifest(path):
    df=read_table(path).copy()
    idc=resolve_col(df,["image_id","image","image_name","filename","file_name","source_image_name"],"manifest id")
    sc=resolve_col(df,["split","set","partition","dataset_split"],"split")
    ic=resolve_col(df,["official_row_index","row_index","index"],"row index",False)
    out=pd.DataFrame({
        "record_key":df[idc].map(norm_key),
        "image_id":df[idc].map(image_name),
        "split":df[sc].astype(str).str.strip().str.casefold().replace({"val":"validation","valid":"validation","dev":"validation"}),
        "official_row_index":np.arange(len(df),dtype=int) if ic is None else pd.to_numeric(df[ic],errors="raise").astype(int)
    })
    group_col=resolve_col(
        df,
        ["group","group_id","template_group","template_id","cluster","cluster_id",
         "phash_group","duplicate_group","near_duplicate_group"],
        "bootstrap group",False
    )
    if group_col is not None:
        out["bootstrap_group"]=df[group_col].astype(str)
    if out.record_key.duplicated().any() or out.official_row_index.duplicated().any():
        raise RuntimeError("manifest IDs/indices not unique")
    return out.sort_values("official_row_index",kind="stable").reset_index(drop=True)

# Discover exact metadata by SHA, not filename assumption.
metadata=[]
for p in INPUT_ROOT.rglob("*"):
    try:
        if p.is_file() and p.suffix.lower() in {".json",".jsonl",".ndjson",".csv"} and p.stat().st_size<250_000_000:
            metadata.append(p)
    except OSError: pass
wanted={"labels":LABEL_SHA,"ocr":OCR_SHA,"split":SPLIT_SHA}
found={}
for i,p in enumerate(metadata,1):
    h=sha256_file(p)
    for role,expected in wanted.items():
        if h==expected: found[role]=p
    if i%50==0: print(f"[META SHA] {i}/{len(metadata)}",flush=True)
if set(found)!=set(wanted):
    raise FileNotFoundError(f"Exact current metadata not found by SHA: {found}")
print("[METADATA PASS]",{k:str(v) for k,v in found.items()})

labels=parse_labels(found["labels"]); ocr=parse_ocr(found["ocr"]); manifest=parse_manifest(found["split"])
master=manifest.merge(labels,on="record_key",validate="one_to_one").merge(ocr,on="record_key",validate="one_to_one")
if len(master)!=9318: raise RuntimeError(f"aligned rows={len(master)}")
counts={k:int(v) for k,v in master.split.value_counts().to_dict().items()}
if counts!={"train":7454,"validation":932,"test":932}: raise RuntimeError(f"split drift {counts}")

# Release count audit.
expected_counts={
 "offensive":{"No":6934,"Yes":2384},
 "vulgar":{"No":6951,"Yes":2367},
 "abuse":{"No":8133,"Yes":1185},
 "sarcastic":{"No":2586,"Yes":6732},
 "language":{"Bangla":6047,"English":1811,"Banglish":1460},
}
for t,e in expected_counts.items():
    got={k:int(v) for k,v in master[t].value_counts().to_dict().items()}
    if got!=e: raise RuntimeError(f"{t} release-count drift got={got} expected={e}")

# Ordered canonical row SHA.
payload="\n".join(f"{int(r.official_row_index)}\t{r.image_id}\t{r.split}" for r in master.sort_values("official_row_index").itertuples())+"\n"
row_sha=hashlib.sha256(payload.encode()).hexdigest()
if row_sha!=CANONICAL_ROW_SHA: raise RuntimeError(f"canonical row SHA drift {row_sha}")

# Ordered TEST identity.
test_payload=[]
for r in master[master.split.eq("test")].sort_values("official_row_index").to_dict("records"):
    tar=[x for x in TARGET_LABELS if x in set(r["target_list"])]
    test_payload.append({
        "sample_id":str(int(r["official_row_index"])),"image":Path(str(r["image_id"])).name.lower(),
        "offensive":r["offensive"],"vulgar":r["vulgar"],"abuse":r["abuse"],"sarcastic":r["sarcastic"],
        "target":tar if tar else ["None"],"language":r["language"],
    })
test_sha=hashlib.sha256(json.dumps(test_payload,ensure_ascii=False,sort_keys=True,separators=(",",":")).encode()).hexdigest()
if test_sha!=TEST_IDENTITY_SHA: raise RuntimeError(f"TEST identity drift {test_sha}")

# Image resolution.
valid_ext={".jpg",".jpeg",".png",".webp",".bmp"}; by_key={}
for p in INPUT_ROOT.rglob("*"):
    try:
        if p.is_file() and p.suffix.casefold() in valid_ext:
            by_key.setdefault(norm_key(p.name),[]).append(p)
    except OSError: pass

image_map={}
for key,name in zip(master.record_key,master.image_id):
    cand=by_key.get(key,[])
    exact=[p for p in cand if p.name.casefold()==Path(str(name)).name.casefold()]
    chosen=exact[0] if len(exact)==1 else (cand[0] if len(cand)==1 else None)
    if chosen is None: raise RuntimeError(f"missing/ambiguous image {name}: {[str(x) for x in cand[:5]]}")
    image_map[key]=str(chosen)
master["image_path"]=master.record_key.map(image_map)

# Full canonical image bundle SHA.
h=hashlib.sha256()
req=master.sort_values("image_id")[["image_id","image_path"]].drop_duplicates().to_dict("records")
for i,r in enumerate(req,1):
    name=Path(str(r["image_id"])).name
    h.update(name.encode()+b"\0"+sha256_file(r["image_path"]).encode()+b"\n")
    if i%250==0 or i==len(req): print(f"[IMAGE SHA] {i}/{len(req)}",flush=True)
image_sha=h.hexdigest()
if image_sha!=IMAGE_BUNDLE_SHA: raise RuntimeError(f"image bundle drift {image_sha}")

# Current release hierarchy.
viol=master[(master.abuse=="Yes")&(master.offensive!="Yes")]
if len(viol): raise RuntimeError(f"Abuse=>Offensive violated in {len(viol)} rows")
tr=master[master.split.eq("train")]
offpos=tr[tr.offensive.eq("Yes")]
hierarchy_counts={
    "train_n":len(tr),
    "offensive_yes_train":int(len(offpos)),
    "abuse_yes_within_offensive_yes":int((offpos.abuse=="Yes").sum()),
    "abuse_no_within_offensive_yes":int((offpos.abuse=="No").sum()),
}
print("[CURRENT RELEASE HIERARCHY]",hierarchy_counts)

# Locked TEST supports.
td=master[master.split.eq("test")]
for t in BINARY_TASKS:
    got={k:int(v) for k,v in td[t].value_counts().to_dict().items()}
    if got!=EXPECTED_SUPPORTS[t]: raise RuntimeError(f"TEST support drift {t}: {got}")
got={k:int(v) for k,v in td.language.value_counts().to_dict().items()}
if got!=EXPECTED_SUPPORTS["language"]: raise RuntimeError(f"TEST language support drift {got}")
got={lab:int(td.target_list.map(lambda x:lab in x).sum()) for lab in TARGET_LABELS}
if got!=EXPECTED_SUPPORTS["target"]: raise RuntimeError(f"TEST target support drift {got}")


master["target_json"]=master.target_list.map(lambda x:json.dumps(x,separators=(",",":")))




test_manifest=master[master.split.eq("test")].sort_values("official_row_index",kind="stable").copy()
TEST_MANIFEST=OUTPUT_ROOT/"prepared_manifest_FINAL_TEST.parquet"
test_manifest[[
    "official_row_index","image_id","image_path","split","cleaned_ocr",
    *BINARY_TASKS,"language","target_json"
]].to_parquet(TEST_MANIFEST,index=False)

print("[CANONICAL LOCKED TEST AUDIT PASS]",{
    "rows":len(test_manifest),
    "ordered_test_sha256":test_sha,
    "image_bundle_sha256":image_sha,
    "manifest":str(TEST_MANIFEST),
    "TEST_model_forwards_before_open":0,
})


In [ ]:
# CELL 5 — PREFETCH EXACT PINNED PANGEA REVISION ONCE; TEST STILL UNOPENED
import os
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_DIR=Path("/kaggle/temp")/f"pangea_locked_test_{MODEL_REVISION[:12]}"
MODEL_DIR.mkdir(parents=True,exist_ok=True)
token=None
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token=os.environ.get("HF_TOKEN")

if not any(MODEL_DIR.iterdir()):
    print("[MODEL SNAPSHOT] downloading pinned revision",MODEL_ID,MODEL_REVISION,flush=True)
    snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
        token=token,
        local_dir_use_symlinks=False,
    )

if not (MODEL_DIR/"config.json").is_file():
    raise RuntimeError("Pinned Pangea snapshot lacks config.json")
print("[PINNED PANGEA SNAPSHOT PASS]",{
    "model_id":MODEL_ID,
    "revision":MODEL_REVISION,
    "snapshot":str(MODEL_DIR),
    "TEST_model_forwards_before_open":0,
})


In [ ]:
# CELL 6 — STRICT FROZEN SPECIALIST PREFLIGHT + LOCKED-TEST WORKER MATERIALIZATION
import json, hashlib, os, torch, re
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

class ResidualFacetAdapter(nn.Module):
    def __init__(self, hidden=1024, bottleneck=256, dropout=0.10):
        super().__init__()
        self.norm=nn.LayerNorm(hidden)
        self.block=nn.Sequential(
            nn.Linear(hidden,bottleneck), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(bottleneck,hidden), nn.Dropout(dropout),
        )
    def forward(self,x):
        return x + self.block(self.norm(x))

class FacetGateTarget(nn.Module):
    def __init__(self):
        super().__init__()
        D=1024
        self.product_norm=nn.LayerNorm(D)
        self.difference_norm=nn.LayerNorm(D)
        self.target_view_gate=nn.Sequential(nn.Linear(4*D,128),nn.GELU(),nn.Linear(128,4))
        self.target_expert_shared=nn.Sequential(nn.Linear(D,D),nn.GELU(),nn.Dropout(0.10))
        self.target_expert_adapter=ResidualFacetAdapter(D,256,0.10)
        self.target_expert_head=nn.Linear(D,7)
        self.target_selector=nn.Sequential(nn.Linear(6,16),nn.GELU(),nn.Linear(16,1))

    def forward(self,v,t,J,zG):
        if v.shape[-1]!=1024 or t.shape[-1]!=1024 or J.shape[-1]!=1024 or zG.shape[-1]!=7:
            raise RuntimeError(f"Facet interface shape drift: v={v.shape} t={t.shape} J={J.shape} zG={zG.shape}")
        prod=self.product_norm(v*t)
        diff=self.difference_norm(torch.abs(v-t))
        views=torch.stack([v,t,prod,diff],dim=1)
        gate_in=torch.cat([v,t,prod,diff],dim=-1)
        view_weights=F.softmax(self.target_view_gate(gate_in.float()),dim=-1)
        h=(views.float()*view_weights.unsqueeze(-1)).sum(dim=1)
        zH=self.target_expert_head(self.target_expert_adapter(self.target_expert_shared(h)))
        pG=torch.sigmoid(zG.float()); pH=torch.sigmoid(zH.float())
        d_it=(1.0-F.cosine_similarity(v.float(),t.float(),dim=-1)).clamp(0,2).unsqueeze(-1).expand(-1,7)
        sf=torch.stack([pG,pH,(pH-pG).abs(),(pG-.5).abs(),(pH-.5).abs(),d_it],dim=-1)
        selector_logit=self.target_selector(sf.reshape(-1,6)).reshape(-1,7)
        selector=torch.sigmoid(selector_logit)
        zStar=zG.float()+selector*(zH.float()-zG.float())
        return zStar,{
            "zH":zH,
            "selector_logit":selector_logit,
            "selector":selector,
            "view_weights":view_weights,
            "image_text_disagreement":d_it[:,0],
        }

# Strict specialist state preflight before TEST opens.
_frozen_state=torch.load(FACET_SELECTED_CHECKPOINT,map_location="cpu",weights_only=False)
if int(_frozen_state.get("step",-1))!=EXPECTED_FACETGATE_STEP:
    raise RuntimeError("Frozen FacetGate selected step drift")
_facet_smoke=FacetGateTarget().cpu()
_facet_smoke.load_state_dict(_frozen_state["facetgate_state"],strict=True)
_facet_smoke.eval()
with torch.no_grad():
    _z,_d=_facet_smoke(
        torch.randn(2,1024),torch.randn(2,1024),torch.randn(2,1024),torch.randn(2,7)
    )
if tuple(_z.shape)!=(2,7):
    raise RuntimeError("FacetGate frozen-state synthetic smoke failed")
del _z,_d,_facet_smoke,_frozen_state
print("[FROZEN FACETGATE CPU FP32 PREFLIGHT PASS]",{
    "seed":RUN_SEED,
    "selected_step":EXPECTED_FACETGATE_STEP,
    "checkpoint_sha256":FACET_SELECTED_CHECKPOINT_SHA256,
    "TEST_model_forwards_before_open":0,
})

WORKER_SOURCE='from __future__ import annotations\nimport os, json, hashlib, random, time\nfrom pathlib import Path\n\n# Lightweight bootstrap executes before any CUDA/model import.\nCFG_PATH=Path(os.environ["PANGEA_LOCKED_TEST_CONFIG"])\nCFG=json.loads(CFG_PATH.read_text(encoding="utf-8"))\nREQUIRED_CFG_KEYS={\n    "seed","session_start_epoch","test_soft_stop_seconds","training_soft_stop_seconds",\n    "output_root","model_dir","parent_checkpoint","feature_dir","prepared_manifest",\n    "protocol_sha256","model_id","model_revision","test_identity_sha256",\n    "parent_fingerprint","feature_binding_sha256","task_instruction","build_id"\n}\n_missing=sorted(REQUIRED_CFG_KEYS-set(CFG))\nif _missing:\n    raise RuntimeError(f"Locked-TEST worker config missing required keys: {_missing}")\nif str(CFG["build_id"])!="PANGEA_LOCKED_TEST_FIXED_20260913_A":\n    raise RuntimeError(f"Locked-TEST build-id mismatch: {CFG[\'build_id\']!r}")\nif float(CFG["test_soft_stop_seconds"])!=float(CFG["training_soft_stop_seconds"]):\n    raise RuntimeError("Locked-TEST soft-stop alias mismatch")\nRANK=int(os.environ["FACET_TEST_RANK"]); WORLD=2\nif RANK not in (0,1):\n    raise RuntimeError(f"Invalid feature rank={RANK}")\nSEED=int(CFG["seed"])\nSESSION_START=float(CFG["session_start_epoch"])\nTEST_SOFT_STOP_SECONDS=float(CFG["test_soft_stop_seconds"])\nTEST_STOP_REQUEST=Path(CFG["output_root"])/"TEST_STOP_REQUEST"\n\nif os.environ.get("FACET_TEST_PREFLIGHT_ONLY","0")=="1":\n    print("[WORKER BOOTSTRAP PREFLIGHT PASS] "+json.dumps({\n        "build_id":CFG["build_id"],\n        "seed":SEED,\n        "rank":RANK,\n        "soft_stop_seconds":TEST_SOFT_STOP_SECONDS,\n        "config_key_count":len(CFG),\n    },sort_keys=True),flush=True)\n    raise SystemExit(0)\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom PIL import Image, ImageFile\nfrom peft import PeftModel\nfrom transformers import AutoProcessor, BitsAndBytesConfig, LlavaNextForConditionalGeneration\n\nImageFile.LOAD_TRUNCATED_IMAGES=False\nDEVICE=torch.device("cuda:0")\ntorch.cuda.set_device(0)\ndef elapsed():\n    return max(0.0,time.time()-SESSION_START)\ndef stop_requested():\n    return TEST_STOP_REQUEST.exists() or elapsed()>=TEST_SOFT_STOP_SECONDS\nos.environ["PYTHONHASHSEED"]=str(SEED)\nrandom.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)\nif hasattr(torch,"set_float32_matmul_precision"): torch.set_float32_matmul_precision("high")\n\n\nBINARY=["offensive","vulgar","abuse","sarcastic"]\nLANGS=["Bangla","English","Banglish"]\nTARGETS=["Individual","Gender","Political","Social","National","Religious","Others"]\n\nclass ResidualAdapter(nn.Module):\n    def __init__(self,dim,hidden,dropout):\n        super().__init__()\n        self.norm=nn.LayerNorm(dim)\n        self.net=nn.Sequential(\n            nn.Linear(dim,hidden),nn.GELU(),nn.Dropout(dropout),\n            nn.Linear(hidden,dim),nn.Dropout(dropout),\n        )\n    def forward(self,x):\n        return x+self.net(self.norm(x))\n\nclass GatedPangeaClassifier(nn.Module):\n    def __init__(self,backbone,hidden=3584):\n        super().__init__()\n        self.backbone=backbone\n        matches=[n for n,m in backbone.named_modules() if m.__class__.__name__=="LlavaNextModel"]\n        if len(matches)!=1: raise RuntimeError(f"Expected one LlavaNextModel, observed={matches}")\n        self._pangea_core_name=matches[0]\n        d=1024; sh=1024; drop=0.10\n        self.image_projection=nn.Sequential(nn.LayerNorm(hidden),nn.Linear(hidden,d),nn.GELU())\n        self.text_projection=nn.Sequential(nn.LayerNorm(hidden),nn.Linear(hidden,d),nn.GELU())\n        self.missing_vision=nn.Parameter(torch.zeros(d),requires_grad=False)\n        self.missing_text=nn.Parameter(torch.zeros(d),requires_grad=False)\n        self.gate=nn.Linear(2*d,d)\n        self.fusion_norm=nn.LayerNorm(d)\n        self.shared=nn.Sequential(nn.Linear(d,sh),nn.GELU(),nn.Dropout(drop))\n        self.adapters=nn.ModuleDict({t:ResidualAdapter(sh,256,drop) for t in BINARY+["language","target"]})\n        self.binary_heads=nn.ModuleDict({t:nn.Linear(sh,1) for t in BINARY})\n        self.language_head=nn.Linear(sh,len(LANGS))\n        self.target_head=nn.Linear(sh,len(TARGETS))\n    def forward(self,**inputs):\n        inputs=dict(inputs)\n        special=inputs.pop("special_token_mask").bool()\n        image_token_mask=inputs.pop("image_token_mask").bool()\n        text_pool_mask=inputs.pop("text_pool_mask").bool()\n        attention=inputs["attention_mask"].bool()\n        core=self.backbone.get_submodule(self._pangea_core_name)\n        out=core(**inputs,use_cache=False)\n        h=out.last_hidden_state\n        image_mask=image_token_mask & attention\n        text_mask=text_pool_mask & attention & (~special)\n        if not image_mask.any(dim=1).all(): raise RuntimeError("No expanded Pangea image tokens.")\n        if not text_mask.any(dim=1).all(): raise RuntimeError("No exact instruction/OCR pool tokens.")\n        def masked_mean(x,m):\n            return torch.stack([x[i][m[i]].float().mean(0) for i in range(x.shape[0])],0)\n        v=self.image_projection(masked_mean(h,image_mask))\n        t=self.text_projection(masked_mean(h,text_mask))\n        g=torch.sigmoid(self.gate(torch.cat([v,t],-1)))\n        J=self.fusion_norm(g*v+(1.0-g)*t)\n        shared=self.shared(J)\n        reps={k:self.adapters[k](shared) for k in self.adapters}\n        logits={k:self.binary_heads[k](reps[k]).squeeze(-1) for k in BINARY}\n        logits["language"]=self.language_head(reps["language"])\n        logits["target"]=self.target_head(reps["target"])\n        return logits,{"v":v,"t":t,"J":J,"gate":g}\n\ndef load_discriminative(model,path):\n    state=torch.load(path,map_location="cpu",weights_only=True)\n    if isinstance(state,dict) and isinstance(state.get("discriminative"),dict):\n        state=state["discriminative"]\n    elif isinstance(state,dict) and isinstance(state.get("state_dict"),dict):\n        state=state["state_dict"]\n    if not isinstance(state,dict) or not all(torch.is_tensor(v) for v in state.values()):\n        raise RuntimeError("classifier_heads.pt is not an exact tensor mapping")\n\n    current={}\n    for name,param in model.named_parameters(recurse=False): current[name]=param\n    for name,buf in model.named_buffers(recurse=False): current[name]=buf\n    for child_name,child in model.named_children():\n        if child_name=="backbone": continue\n        for k,v in child.state_dict().items(): current[f"{child_name}.{k}"]=v\n\n    expected=set(current); observed=set(state)\n    missing=sorted(expected-observed); unexpected=sorted(observed-expected)\n    if missing or unexpected:\n        raise RuntimeError(\n            f"Discriminative state mismatch: missing={missing[:20]} unexpected={unexpected[:20]}"\n        )\n    for k in expected:\n        if tuple(current[k].shape)!=tuple(state[k].shape):\n            raise RuntimeError(\n                f"Discriminative tensor shape mismatch key={k}: expected={tuple(current[k].shape)} observed={tuple(state[k].shape)}"\n            )\n    with torch.no_grad():\n        direct_params=dict(model.named_parameters(recurse=False))\n        direct_buffers=dict(model.named_buffers(recurse=False))\n        for k,dst in {**direct_params,**direct_buffers}.items():\n            dst.copy_(state[k].to(device=dst.device,dtype=dst.dtype))\n        for child_name,child in model.named_children():\n            if child_name=="backbone": continue\n            prefix=child_name+"."\n            sub={k[len(prefix):]:v for k,v in state.items() if k.startswith(prefix)}\n            child.load_state_dict(sub,strict=True)\n    return sorted(state)\n\nMODEL_DIR=Path(CFG["model_dir"]); PARENT=Path(CFG["parent_checkpoint"])\nprocessor=AutoProcessor.from_pretrained(str(MODEL_DIR),local_files_only=True,trust_remote_code=False)\nif processor.__class__.__name__!="LlavaNextProcessor":\n    raise RuntimeError(f"Unexpected processor={processor.__class__.__name__}")\nprocessor.tokenizer.padding_side="right"\nif int(getattr(processor,"patch_size",-1))!=14:\n    raise RuntimeError(f"Pangea LlavaNextProcessor patch_size drift: {getattr(processor,\'patch_size\',None)}")\nif getattr(processor,"vision_feature_select_strategy",None) is None:\n    processor.vision_feature_select_strategy="default"\nif processor.vision_feature_select_strategy!="default":\n    raise RuntimeError(f"Pangea vision_feature_select_strategy drift: {processor.vision_feature_select_strategy}")\nprocessor.num_additional_image_tokens=1\nif int(getattr(processor,"num_additional_image_tokens",-1))!=1:\n    raise RuntimeError("Pangea num_additional_image_tokens drift")\nimage_token_id=int(processor.tokenizer.convert_tokens_to_ids("<image>"))\nif image_token_id!=151646: raise RuntimeError(f"Pangea image token id drift: {image_token_id}")\ncrop=getattr(processor.image_processor,"crop_size",None)\nch=getattr(crop,"height",None); cw=getattr(crop,"width",None)\nif (ch is None or cw is None) and hasattr(crop,"get"):\n    ch=crop.get("height",ch); cw=crop.get("width",cw)\ntry: crop_hw=(int(ch),int(cw))\nexcept (TypeError,ValueError): crop_hw=None\nif crop_hw!=(336,336): raise RuntimeError(f"Pangea native crop geometry drift: {crop!r} parsed={crop_hw}")\nsize=getattr(processor.image_processor,"size",None)\nshortest=getattr(size,"shortest_edge",None)\nif shortest is None and hasattr(size,"get"): shortest=size.get("shortest_edge",None)\nif int(shortest or -1)!=336: raise RuntimeError(f"Pangea native shortest_edge drift: {size!r}")\n\n# Independent per-GPU worker: no NCCL collectives are used for feature export.\n# The controller starts only ranks whose validated shard is still missing.\nquant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",\n                          bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)\nbase=LlavaNextForConditionalGeneration.from_pretrained(\n    str(MODEL_DIR),local_files_only=True,quantization_config=quant,\n    torch_dtype=torch.float16,device_map={"":0},low_cpu_mem_usage=True,\n    attn_implementation="sdpa",\n)\nbase.config.use_cache=False\nif hasattr(base.config,"text_config"): base.config.text_config.use_cache=False\nif int(base.config.text_config.hidden_size)!=3584: raise RuntimeError("Pangea hidden size drift")\nif int(base.config.image_token_index)!=151646: raise RuntimeError(f"Pangea image-token config drift: {base.config.image_token_index}")\nif len(processor.tokenizer)>int(base.get_input_embeddings().num_embeddings):\n    raise RuntimeError("Tokenizer larger than Pangea embedding table")\nbackbone=PeftModel.from_pretrained(base,str(PARENT/"adapter"),is_trainable=False)\n(Path(CFG["feature_dir"])/f"locked_test_worker_rank{RANK}_MODEL_READY").write_text("ready\\n",encoding="utf-8")\nmodel=GatedPangeaClassifier(backbone,3584)\nfor name,child in model.named_children():\n    if name!="backbone": child.to(DEVICE)\nmodel.missing_vision.data=model.missing_vision.data.to(DEVICE)\nmodel.missing_text.data=model.missing_text.data.to(DEVICE)\nloaded_keys=load_discriminative(model,PARENT/"classifier_heads.pt")\nmodel.eval()\nfor p in model.parameters(): p.requires_grad_(False)\n\ntask_instruction=CFG["task_instruction"]\ndef truncate_ocr(text):\n    ids=processor.tokenizer(str(text),add_special_tokens=False,truncation=True,max_length=384)["input_ids"]\n    return processor.tokenizer.decode(ids,skip_special_tokens=True)\n\ndef encode(row):\n    with Image.open(row["image_path"]) as im: image=im.convert("RGB").copy()\n    ocr=truncate_ocr(row["cleaned_ocr"])\n    suffix=ocr if ocr.strip() else "[OCR EMPTY—USE IMAGE]"\n    task_text=task_instruction+"Cleaned OCR text: "+suffix\n    messages=[{"role":"user","content":"<image>\\n"+task_text}]\n    serialized=processor.tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=False)\n    inputs=processor(images=image,text=serialized,return_tensors="pt")\n    inputs.pop("labels",None); inputs.pop("token_type_ids",None)\n    required={"input_ids","attention_mask","pixel_values","image_sizes"}\n    if not required.issubset(inputs): raise RuntimeError(f"Processor keys drift: {sorted(inputs)}")\n    image_mask=inputs["input_ids"].eq(image_token_id)\n    special=torch.zeros_like(inputs["input_ids"],dtype=torch.bool)\n    for sid in processor.tokenizer.all_special_ids: special |= inputs["input_ids"].eq(int(sid))\n    raw=processor.tokenizer(serialized,add_special_tokens=False,return_offsets_mapping=True)\n    raw_ids=[int(x) for x in raw["input_ids"]]; offsets=[tuple(map(int,x)) for x in raw["offset_mapping"]]\n    content_start=serialized.rfind(task_text)\n    if content_start<0: raise RuntimeError("Exact frozen task_text bytes not found in Pangea serialized prompt")\n    content_end=content_start+len(task_text)\n    raw_img=[i for i,x in enumerate(raw_ids) if x==image_token_id]\n    if len(raw_img)!=1: raise RuntimeError(f"Expected one raw <image>, got {len(raw_img)}")\n    ri=raw_img[0]\n    exp_ids=[int(x) for x in inputs["input_ids"][0].tolist()]\n    exp_img=[i for i,x in enumerate(exp_ids) if x==image_token_id]\n    if not exp_img: raise RuntimeError("No expanded image positions")\n    first,last=exp_img[0],exp_img[-1]\n    if exp_img!=list(range(first,last+1)): raise RuntimeError("Expanded image positions noncontiguous")\n    if raw_ids[:ri]!=exp_ids[:first] or raw_ids[ri+1:]!=exp_ids[last+1:]:\n        raise RuntimeError("Non-image tokenization changed during any-resolution expansion")\n    special_ids=set(map(int,processor.tokenizer.all_special_ids))\n    raw_keep=[]\n    for tok,(a,b) in zip(raw_ids,offsets):\n        raw_keep.append(bool(b>content_start and a<content_end and tok not in special_ids))\n    if any(raw_keep[:ri+1]):\n        raise RuntimeError("Frozen task-text mask includes native system/role/pre-image scaffolding")\n    suffix_keep=raw_keep[ri+1:]\n    if len(suffix_keep)!=len(exp_ids[last+1:]):\n        raise RuntimeError("Frozen text-mask suffix length drift after Pangea image expansion")\n    text_pool=torch.zeros_like(inputs["input_ids"],dtype=torch.bool)\n    for j,keep in enumerate(suffix_keep):\n        if keep: text_pool[0,last+1+j]=True\n    text_pool &= inputs["attention_mask"].bool()\n    text_pool &= ~image_mask\n    text_pool &= ~special\n    if (text_pool & image_mask).any(): raise RuntimeError("Frozen text-pool mask overlaps image-token positions")\n    if int(text_pool.sum())<=0: raise RuntimeError("Exact text pool is empty")\n    if tuple(inputs["pixel_values"].shape[-2:])!=(336,336):\n        raise RuntimeError(f"Pangea native pixel patch geometry drift: {tuple(inputs[\'pixel_values\'].shape)}")\n    inputs["image_token_mask"]=image_mask\n    inputs["special_token_mask"]=special\n    inputs["text_pool_mask"]=text_pool\n    moved={}\n    for k,v in inputs.items():\n        if torch.is_tensor(v):\n            moved[k]=v.to(DEVICE,dtype=torch.float16 if k=="pixel_values" else v.dtype,non_blocking=True)\n        else: moved[k]=v\n    if not torch.isfinite(moved["pixel_values"]).all(): raise RuntimeError("Pangea processor returned non-finite pixel_values")\n    return moved\n\nframe=pd.read_parquet(CFG["prepared_manifest"]).sort_values("official_row_index").reset_index(drop=True)\ndev=frame[frame["split"].eq("test")].sort_values("official_row_index").reset_index(drop=True)\nlocal=dev.iloc[RANK::WORLD].copy().reset_index(drop=True)\nout=Path(CFG["feature_dir"]); out.mkdir(parents=True,exist_ok=True)\npartial_path=out/f"pangea_locked_test_rank{RANK}_PARTIAL.npz"\npartial_meta=out/f"pangea_locked_test_rank{RANK}_PARTIAL.json"\n\nidx=[]; splits=[]; V=[]; T=[]; J=[]; ZB=[]; ZL=[]; ZT=[]\ncompleted=0\nif partial_path.is_file():\n    pdx=np.load(partial_path,allow_pickle=False)\n    idx=pdx["official_row_index"].astype(np.int64).tolist()\n    splits=pdx["split"].astype(str).tolist()\n    V=list(pdx["visual"].astype(np.float32)); T=list(pdx["text"].astype(np.float32)); J=list(pdx["joint"].astype(np.float32))\n    ZB=list(pdx["z_binary"].astype(np.float32)); ZL=list(pdx["z_language"].astype(np.float32)); ZT=list(pdx["z_target"].astype(np.float32))\n    completed=len(idx)\n    expected=local.iloc[:completed]["official_row_index"].astype(int).tolist()\n    if idx!=expected:\n        raise RuntimeError(f"Partial feature shard identity drift rank={RANK}: completed={completed}")\n    print(f"[LOCKED TEST RESUME rank={RANK}] completed={completed}/{len(local)}",flush=True)\n\ndef save_partial(reason):\n    np.savez_compressed(partial_path,\n        official_row_index=np.asarray(idx,dtype=np.int64),\n        split=np.asarray(splits,dtype="<U10"),\n        visual=np.asarray(V,dtype=np.float32),\n        text=np.asarray(T,dtype=np.float32),\n        joint=np.asarray(J,dtype=np.float32),\n        z_binary=np.asarray(ZB,dtype=np.float32),\n        z_language=np.asarray(ZL,dtype=np.float32),\n        z_target=np.asarray(ZT,dtype=np.float32),\n    )\n    meta={\n        "schema":"M3Offend_Pangea_FacetGate_LOCKED_TEST_feature_shard_partial",\n        "status":"PARTIAL_RESUMABLE","reason":str(reason),\n        "protocol_sha256":CFG["protocol_sha256"],"rank":RANK,"world":WORLD,\n        "rows_completed":len(idx),"rows_total":len(local),\n        "model_id":CFG["model_id"],"model_revision":CFG["model_revision"],"seed":SEED,\n        "ordered_test_identity_sha256":CFG["test_identity_sha256"],"locked_test_authorized":True,\n        "parent_checkpoint_fingerprint":CFG["parent_fingerprint"],\n        "feature_binding_sha256":CFG["feature_binding_sha256"],\n        "elapsed_seconds":elapsed(),"test_model_forwards":len(idx),\n    }\n    partial_meta.write_text(json.dumps(meta,indent=2),encoding="utf-8")\n\nwith torch.no_grad():\n    for n in range(completed,len(local)):\n        if n==completed or n%25==0:\n            if stop_requested():\n                save_partial("11h00_locked_test_soft_stop")\n                raise SystemExit(0)\n        row=local.iloc[n]\n        inp=encode(row)\n        with torch.autocast("cuda",dtype=torch.float16):\n            logits,diag=model(**inp)\n        idx.append(int(row["official_row_index"])); splits.append(str(row["split"]))\n        V.append(diag["v"].float().cpu().numpy()[0]); T.append(diag["t"].float().cpu().numpy()[0])\n        J.append(diag["J"].float().cpu().numpy()[0])\n        ZB.append(np.asarray([float(logits[t].float().cpu()[0]) for t in BINARY],dtype=np.float32))\n        ZL.append(logits["language"].float().cpu().numpy()[0])\n        ZT.append(logits["target"].float().cpu().numpy()[0])\n        if (n+1)%50==0 or n+1==len(local):\n            print(f"[LOCKED TEST rank={RANK}] {n+1}/{len(local)} elapsed_h={elapsed()/3600:.2f}",flush=True)\n            if n+1 < len(local):\n                save_partial("periodic_progress_checkpoint")\n\nnp.savez_compressed(out/f"pangea_locked_test_rank{RANK}.npz",\n    official_row_index=np.asarray(idx,dtype=np.int64),\n    split=np.asarray(splits,dtype="<U10"),\n    visual=np.asarray(V,dtype=np.float32),\n    text=np.asarray(T,dtype=np.float32),\n    joint=np.asarray(J,dtype=np.float32),\n    z_binary=np.asarray(ZB,dtype=np.float32),\n    z_language=np.asarray(ZL,dtype=np.float32),\n    z_target=np.asarray(ZT,dtype=np.float32),\n)\nmeta={\n    "schema":"M3Offend_Pangea_FacetGate_LOCKED_TEST_feature_shard",\n    "protocol_sha256":CFG["protocol_sha256"],"rank":RANK,"world":WORLD,"rows":len(idx),\n    "model_id":CFG["model_id"],"model_revision":CFG["model_revision"],"seed":SEED,\n    "ordered_test_identity_sha256":CFG["test_identity_sha256"],"locked_test_authorized":True,\n    "parent_checkpoint_fingerprint":CFG["parent_fingerprint"],\n    "feature_binding_sha256":CFG["feature_binding_sha256"],"representation_dim":1024,\n    "test_model_forwards":len(idx),\n}\n(out/f"pangea_locked_test_rank{RANK}.json").write_text(json.dumps(meta,indent=2),encoding="utf-8")\nfor p in [partial_path,partial_meta]:\n    if p.exists(): p.unlink()\n'
WORKER_PATH=OUTPUT_ROOT/"pangea_facetgate_single_seed_locked_test_worker.py"
WORKER_PATH.write_text(WORKER_SOURCE,encoding="utf-8")
compile(WORKER_SOURCE,str(WORKER_PATH),"exec")
WORKER_SHA256=hashlib.sha256(WORKER_SOURCE.encode("utf-8")).hexdigest()

WORKER_CFG={
    "seed":RUN_SEED,
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "model_dir":str(MODEL_DIR),
    "parent_checkpoint":str(PARENT_CHECKPOINT_DIR),
    "prepared_manifest":str(TEST_MANIFEST),
    "feature_dir":str(FEATURE_DIR),
    "parent_fingerprint":PARENT_CHECKPOINT_FINGERPRINT,
    "feature_binding_sha256":PANGEA_FEATURE_BINDING_SHA256,
    "protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256,
    "test_identity_sha256":TEST_IDENTITY_SHA,
    "task_instruction":TASK_INSTRUCTION,
    "session_start_epoch":time.time(),
    "test_soft_stop_seconds":TEST_SOFT_STOP_SECONDS,
    "training_soft_stop_seconds":TEST_SOFT_STOP_SECONDS,
    "build_id":LOCKED_TEST_BUILD_ID,
    "output_root":str(OUTPUT_ROOT),
}
WORKER_CFG_PATH=OUTPUT_ROOT/"pangea_locked_test_worker_config.json"
WORKER_CFG_PATH.write_text(json.dumps(WORKER_CFG,indent=2),encoding="utf-8")

# Fail-closed worker/config contract audit.
_cfg_roundtrip=json.loads(WORKER_CFG_PATH.read_text(encoding="utf-8"))
if "test_soft_stop_seconds" not in _cfg_roundtrip:
    raise RuntimeError("Locked-TEST worker config missing test_soft_stop_seconds")
if float(_cfg_roundtrip["test_soft_stop_seconds"]) != float(TEST_SOFT_STOP_SECONDS):
    raise RuntimeError("Locked-TEST soft-stop value drift between parent notebook and worker config")
_cfg_refs=set(re.findall(r'CFG\["([^"]+)"\]',WORKER_SOURCE))
_cfg_refs.update(re.findall(r"CFG\['([^']+)'\]",WORKER_SOURCE))
_cfg_missing=sorted(k for k in _cfg_refs if k not in _cfg_roundtrip)
if _cfg_missing:
    raise RuntimeError(f"Locked-TEST worker/config key mismatch: missing={_cfg_missing}")
print("[LOCKED-TEST WORKER CONFIG CONTRACT PASS]",{
    "seed":RUN_SEED,
    "required_key_count":len(_cfg_refs),
    "test_soft_stop_seconds":float(_cfg_roundtrip["test_soft_stop_seconds"]),
    "missing_keys":[],
    "TEST_model_forwards_before_open":0,
})

if _cfg_roundtrip.get("build_id")!=LOCKED_TEST_BUILD_ID:
    raise RuntimeError("Locked-TEST worker config build-id drift")
if float(_cfg_roundtrip["training_soft_stop_seconds"])!=float(_cfg_roundtrip["test_soft_stop_seconds"]):
    raise RuntimeError("Locked-TEST soft-stop aliases disagree")

# Execute the actual materialized worker in config-only mode.
# It exits before importing numpy/torch/transformers, before CUDA, and before TEST access.
import subprocess, sys
_pre_env=os.environ.copy()
_pre_env["FACET_TEST_RANK"]="0"
_pre_env["PANGEA_LOCKED_TEST_CONFIG"]=str(WORKER_CFG_PATH)
_pre_env["FACET_TEST_PREFLIGHT_ONLY"]="1"
_pre_env["CUDA_VISIBLE_DEVICES"]=""
_pre=subprocess.run(
    [sys.executable,"-u",str(WORKER_PATH)],
    env=_pre_env,cwd="/kaggle/working",
    text=True,capture_output=True,timeout=60,
)
if _pre.returncode!=0 or "[WORKER BOOTSTRAP PREFLIGHT PASS]" not in _pre.stdout:
    raise RuntimeError(
        "Materialized locked-TEST worker bootstrap preflight failed. "
        f"rc={_pre.returncode} stdout={_pre.stdout[-2000:]} stderr={_pre.stderr[-2000:]}"
    )
WORKER_BOOTSTRAP_PREFLIGHT_OK=True
print("[LOCKED-TEST MATERIALIZED WORKER BOOTSTRAP PASS]",{
    "seed":RUN_SEED,
    "build_id":LOCKED_TEST_BUILD_ID,
    "worker_sha256":WORKER_SHA256,
    "stdout":_pre.stdout.strip(),
    "TEST_model_forwards_before_open":0,
})

if "torch.distributed" in WORKER_SOURCE or "init_process_group" in WORKER_SOURCE:
    raise RuntimeError("Locked-test worker must not use distributed collectives")
if 'frame[frame["split"].eq("test")]' not in WORKER_SOURCE:
    raise RuntimeError("Locked-test worker is not restricted to TEST split")

print("[LOCKED-TEST WORKER STATIC QA PASS]",{
    "worker_sha256":WORKER_SHA256,
    "independent_per_gpu":True,
    "nccl_collectives":False,
    "expected_rows_total":932,
    "expected_rows_per_rank":466,
    "TEST_model_forwards_before_open":0,
})

In [ ]:
# CELL 7 — OPEN LOCKED TEST ONCE FOR THIS FROZEN SEED
# Only this cell performs Pangea TEST model forwards.
if not globals().get("WORKER_BOOTSTRAP_PREFLIGHT_OK",False):
    raise RuntimeError("Refusing locked TEST: actual materialized worker bootstrap preflight did not pass.")
if LOCKED_TEST_BUILD_ID!="PANGEA_LOCKED_TEST_FIXED_20260913_A":
    raise RuntimeError(f"Unexpected locked-TEST build id: {LOCKED_TEST_BUILD_ID}")
import os, sys, time, json, shutil, zipfile, subprocess, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch

if not AUTHORIZE_LOCKED_TEST or LOCKED_TEST_ACK!="I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST":
    raise RuntimeError("Explicit final locked-TEST authorization missing.")
if not torch.cuda.is_available() or torch.cuda.device_count()!=2:
    raise RuntimeError(f"Pangea locked TEST requires Kaggle T4x2; cuda={torch.cuda.is_available()} count={torch.cuda.device_count()}")

# Never repeat a completed locked TEST merely because of its score.
completed_prior=[]
for p in INPUT_ROOT.rglob(f"Pangea_FacetGate_Seed{RUN_SEED}_LOCKED_TEST_RESULT.json"):
    try:
        q=json.loads(p.read_text(encoding="utf-8"))
        if (int(q.get("seed",-1))==RUN_SEED and
            q.get("protocol_sha256")==SHARED_FORWARD_PROTOCOL_SHA256 and
            q.get("status")=="COMPLETE"):
            completed_prior.append(str(p))
    except Exception:
        pass
if completed_prior:
    raise RuntimeError(
        "A completed locked-TEST result for this frozen seed is already mounted. "
        "Refusing to rerun TEST. Existing="+json.dumps(completed_prior)
    )

test_df=pd.read_parquet(TEST_MANIFEST).sort_values("official_row_index").reset_index(drop=True)
expected_by_rank={
    r:test_df.iloc[r::2]["official_row_index"].astype(np.int64).to_numpy()
    for r in (0,1)
}

def _meta_ok(meta_path,npz_path,rank,partial):
    try:
        m=json.loads(Path(meta_path).read_text(encoding="utf-8"))
        if int(m.get("seed",-1))!=RUN_SEED: return False
        if m.get("protocol_sha256")!=SHARED_FORWARD_PROTOCOL_SHA256: return False
        if m.get("model_id")!=MODEL_ID or m.get("model_revision")!=MODEL_REVISION: return False
        if m.get("parent_checkpoint_fingerprint")!=PARENT_CHECKPOINT_FINGERPRINT: return False
        if m.get("feature_binding_sha256")!=PANGEA_FEATURE_BINDING_SHA256: return False
        if m.get("ordered_test_identity_sha256")!=TEST_IDENTITY_SHA: return False
        if int(m.get("rank",-1))!=rank or int(m.get("world",-1))!=2: return False
        z=np.load(npz_path,allow_pickle=False)
        idx=z["official_row_index"].astype(np.int64)
        if partial:
            if len(idx)<1 or len(idx)>=len(expected_by_rank[rank]): return False
            if not np.array_equal(idx,expected_by_rank[rank][:len(idx)]): return False
        else:
            if len(idx)!=len(expected_by_rank[rank]): return False
            if not np.array_equal(idx,expected_by_rank[rank]): return False
        if set(z["split"].astype(str))!={"test"}: return False
        for key,shape in {
            "visual":(len(idx),1024),"text":(len(idx),1024),"joint":(len(idx),1024),
            "z_binary":(len(idx),4),"z_language":(len(idx),3),"z_target":(len(idx),7)
        }.items():
            if tuple(z[key].shape)!=shape: return False
        return True
    except Exception:
        return False

# Exact technical-resume import: same seed, same parent, same TEST identity only.
for rank in (0,1):
    final_dest=FEATURE_DIR/f"pangea_locked_test_rank{rank}.npz"
    final_meta=FEATURE_DIR/f"pangea_locked_test_rank{rank}.json"
    part_dest=FEATURE_DIR/f"pangea_locked_test_rank{rank}_PARTIAL.npz"
    part_meta=FEATURE_DIR/f"pangea_locked_test_rank{rank}_PARTIAL.json"
    if _meta_ok(final_meta,final_dest,rank,False):
        continue

    finals=[]
    for mp in INPUT_ROOT.rglob(f"pangea_locked_test_rank{rank}.json"):
        npz=mp.with_suffix(".npz")
        if npz.is_file() and _meta_ok(mp,npz,rank,False):
            finals.append((mp,npz))
    if finals:
        mp,npz=sorted(finals,key=lambda x:(len(str(x[0])),str(x[0])))[0]
        shutil.copy2(mp,final_meta); shutil.copy2(npz,final_dest)
        print("[LOCKED TEST FULL SHARD RESTORED]",{"rank":rank,"source":str(mp)})
        continue

    partials=[]
    for mp in INPUT_ROOT.rglob(f"pangea_locked_test_rank{rank}_PARTIAL.json"):
        npz=mp.with_suffix(".npz")
        if npz.is_file() and _meta_ok(mp,npz,rank,True):
            rows=len(np.load(npz,allow_pickle=False)["official_row_index"])
            partials.append((rows,mp,npz))
    if partials:
        rows,mp,npz=max(partials,key=lambda x:x[0])
        shutil.copy2(mp,part_meta); shutil.copy2(npz,part_dest)
        print("[LOCKED TEST PARTIAL SHARD RESTORED]",{"rank":rank,"rows":rows,"source":str(mp)})

def full_ok(rank):
    return _meta_ok(
        FEATURE_DIR/f"pangea_locked_test_rank{rank}.json",
        FEATURE_DIR/f"pangea_locked_test_rank{rank}.npz",
        rank,False
    )

missing=[r for r in (0,1) if not full_ok(r)]
print("[LOCKED TEST EXECUTION PLAN]",{
    "seed":RUN_SEED,
    "missing_ranks":missing,
    "reused_complete_ranks":[r for r in (0,1) if r not in missing],
    "authorization":"FINAL_LOCKED_TEST",
    "build_id":LOCKED_TEST_BUILD_ID,
})

procs={}
visible=[x.strip() for x in os.environ.get("CUDA_VISIBLE_DEVICES","0,1").split(",") if x.strip()]
if len(visible)<2: visible=["0","1"]

for r in missing:
    env=os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"]=visible[r]
    env["FACET_TEST_RANK"]=str(r)
    env["PANGEA_LOCKED_TEST_CONFIG"]=str(WORKER_CFG_PATH)
    env["TOKENIZERS_PARALLELISM"]="false"
    cmd=[sys.executable,"-u",str(WORKER_PATH)]
    print("[LOCKED TEST WORKER START]",{"rank":r,"gpu":env["CUDA_VISIBLE_DEVICES"],"cmd":" ".join(cmd)},flush=True)
    procs[r]=subprocess.Popen(cmd,env=env,cwd="/kaggle/working")
    if len(missing)>1 and r==missing[0]:
        ready=FEATURE_DIR/f"locked_test_worker_rank{r}_MODEL_READY"
        t0=time.time()
        while not ready.exists() and procs[r].poll() is None and time.time()-t0<1200:
            time.sleep(2)
        if procs[r].poll() not in (None,0):
            break

failures={}
for r,p in procs.items():
    rc=p.wait()
    print("[LOCKED TEST WORKER EXIT]",{"rank":r,"returncode":rc},flush=True)
    if rc!=0: failures[r]=rc

def package_progress(reason):
    zp=Path("/kaggle/working")/f"M3Offend_9318_Pangea7BHF_FacetGate_Seed{RUN_SEED}_LOCKED_TEST_CURRENT_PROGRESS.zip"
    with zipfile.ZipFile(zp,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for p in FEATURE_DIR.rglob("*"):
            if p.is_file():
                z.write(p,arcname=f"locked_test_features/{p.name}")
        for p in [FACET_SELECTED_CHECKPOINT,Path(FACET_STATE["root"])/"validation_thresholds.json",
                  OUTPUT_ROOT/"BASE_GATED_BINDING.json",TEST_MANIFEST,WORKER_CFG_PATH]:
            if Path(p).is_file():
                z.write(p,arcname=Path(p).name)
    print("[LOCKED TEST PROGRESS PACKAGE]",{"reason":str(reason),"path":str(zp)})
    return zp

if failures:
    zp=package_progress(f"worker_failure_{failures}")
    raise RuntimeError(f"Locked TEST worker failure={failures}. Exact technical progress preserved at {zp}")

if not all(full_ok(r) for r in (0,1)):
    zp=package_progress("incomplete_after_worker_exit")
    raise RuntimeError(f"Locked TEST incomplete. Exact technical progress preserved at {zp}")

print("[SINGLE-SEED LOCKED TEST FEATURE EXTRACTION COMPLETE]",{
    "seed":RUN_SEED,
    "rows":932,
    "rank_rows":[466,466],
    "no_nccl_collectives":True,
})


In [ ]:
# CELL 8 — CPU-ONLY FROZEN FACETGATE FINALIZATION + PAPER-READY SINGLE-SEED OUTPUT
# No additional Pangea TEST model forward occurs in this cell.
import json, math, hashlib, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    average_precision_score, hamming_loss, confusion_matrix
)

parts=[]
for rank in (0,1):
    z=np.load(FEATURE_DIR/f"pangea_locked_test_rank{rank}.npz",allow_pickle=False)
    parts.append({k:z[k] for k in z.files})
merged={k:np.concatenate([p[k] for p in parts],axis=0) for k in parts[0]}
order=np.argsort(merged["official_row_index"].astype(np.int64),kind="stable")
merged={k:v[order] for k,v in merged.items()}
if len(merged["official_row_index"])!=932 or len(np.unique(merged["official_row_index"]))!=932:
    raise RuntimeError("Locked TEST feature merge row identity failure")
if set(merged["split"].astype(str))!={"test"}:
    raise RuntimeError("Locked TEST shard contains non-TEST split")

td=pd.read_parquet(TEST_MANIFEST).sort_values("official_row_index").reset_index(drop=True)
if td.official_row_index.astype(np.int64).tolist()!=merged["official_row_index"].astype(np.int64).tolist():
    raise RuntimeError("Locked TEST manifest/feature identity mismatch")

V=merged["visual"].astype(np.float32)
T=merged["text"].astype(np.float32)
J=merged["joint"].astype(np.float32)
ZBIN=merged["z_binary"].astype(np.float32)
ZLANG=merged["z_language"].astype(np.float32)
ZT=merged["z_target"].astype(np.float32)

bin_gold=np.asarray([[1 if str(r[t])=="Yes" else 0 for t in BINARY_TASKS] for r in td.to_dict("records")],dtype=np.int64)
lang_gold=np.asarray([LANGUAGE_LABELS.index(str(x)) for x in td.language],dtype=np.int64)
target_gold=np.zeros((932,7),dtype=np.int64)
for i,s in enumerate(td["target_json"].astype(str)):
    vals=json.loads(s)
    for j,lab in enumerate(TARGET_LABELS):
        target_gold[i,j]=1 if lab in vals else 0

bin_prob=1.0/(1.0+np.exp(-ZBIN.astype(np.float64)))
lang_exp=np.exp(ZLANG.astype(np.float64)-ZLANG.astype(np.float64).max(axis=1,keepdims=True))
lang_prob=lang_exp/lang_exp.sum(axis=1,keepdims=True)
gated_target_prob=1.0/(1.0+np.exp(-ZT.astype(np.float64)))

# Frozen specialist only; checkpoint and thresholds were selected on validation before TEST opened.
facet=FacetGateTarget().cpu()
state=torch.load(FACET_SELECTED_CHECKPOINT,map_location="cpu",weights_only=False)
facet.load_state_dict(state["facetgate_state"],strict=True)
facet.eval()
zs=[]; sels=[]; views=[]; dits=[]
with torch.no_grad():
    for st in range(0,932,64):
        en=min(st+64,932)
        z,diag=facet(
            torch.from_numpy(V[st:en]),
            torch.from_numpy(T[st:en]),
            torch.from_numpy(J[st:en]),
            torch.from_numpy(ZT[st:en]),
        )
        zs.append(z.numpy())
        sels.append(diag["selector"].numpy())
        views.append(diag["view_weights"].numpy())
        dits.append(diag["image_text_disagreement"].numpy())
zstar=np.concatenate(zs,axis=0).astype(np.float64)
facet_prob=1.0/(1.0+np.exp(-zstar))
selector=np.concatenate(sels,axis=0).astype(np.float64)
view_weights=np.concatenate(views,axis=0).astype(np.float64)
image_text_disagreement=np.concatenate(dits,axis=0).astype(np.float64)

gated_target_thr=np.asarray([THRESHOLDS_NORMALIZED["target"][x] for x in TARGET_LABELS],dtype=np.float64)
facet_target_thr=np.asarray([FACET_VALIDATION_THRESHOLDS[x] for x in TARGET_LABELS],dtype=np.float64)

def target_bundle(y,prob,thr):
    pred=(prob>=thr.reshape(1,-1)).astype(np.int64)
    per_f1=[]; per_ap=[]; rows=[]
    for j,lab in enumerate(TARGET_LABELS):
        pr,rc,f1,_=precision_recall_fscore_support(
            y[:,j],pred[:,j],labels=[1],average=None,zero_division=0
        )
        f=float(f1[0]); p=float(pr[0]); r=float(rc[0])
        ap=float(average_precision_score(y[:,j],prob[:,j])) if len(np.unique(y[:,j]))>1 else float("nan")
        support=int(y[:,j].sum())
        per_f1.append(f); per_ap.append(ap)
        rows.append({
            "label":lab,"support":support,"threshold":float(thr[j]),
            "precision":p,"recall":r,"f1":f,"average_precision":ap
        })
    return {
        "macro_f1":float(np.mean(per_f1)),
        "map":float(np.nanmean(per_ap)),
        "micro_f1":float(f1_score(y,pred,average="micro",zero_division=0)),
        "samples_f1":float(f1_score(y,pred,average="samples",zero_division=0)),
        "exact_set_accuracy":float(np.mean(np.all(y==pred,axis=1))),
        "hamming_loss":float(hamming_loss(y,pred)),
        "rare_target_macro_f1":float(np.mean([per_f1[TARGET_LABELS.index("National")],per_f1[TARGET_LABELS.index("Religious")]])),
        "per_label_f1":dict(zip(TARGET_LABELS,per_f1)),
        "per_label_ap":dict(zip(TARGET_LABELS,per_ap)),
        "rows":rows,
        "pred":pred,
    }

gated_target=target_bundle(target_gold,gated_target_prob,gated_target_thr)
facet_target=target_bundle(target_gold,facet_prob,facet_target_thr)

task_acc=[]; task_mf1=[]; binary_rows=[]; task_rows=[]
bin_preds={}
for k,t in enumerate(BINARY_TASKS):
    th=float(THRESHOLDS_NORMALIZED["binary"][t])
    pred=(bin_prob[:,k]>=th).astype(np.int64)
    bin_preds[t]=pred
    acc=float(accuracy_score(bin_gold[:,k],pred))
    mf1=float(f1_score(bin_gold[:,k],pred,average="macro",zero_division=0))
    task_acc.append(acc); task_mf1.append(mf1)
    task_rows.append({"task":t,"accuracy":acc,"macro_f1":mf1,"threshold":th})
    pr,rc,f1,sup=precision_recall_fscore_support(bin_gold[:,k],pred,labels=[0,1],zero_division=0)
    for ci,name in enumerate(["No","Yes"]):
        binary_rows.append({
            "task":t,"class":name,"support":int(sup[ci]),
            "precision":float(pr[ci]),"recall":float(rc[ci]),"f1":float(f1[ci]),
            "threshold":th
        })

lang_pred=lang_prob.argmax(axis=1)
lang_acc=float(accuracy_score(lang_gold,lang_pred))
lang_mf1=float(f1_score(lang_gold,lang_pred,average="macro",zero_division=0))
task_acc.append(lang_acc); task_mf1.append(lang_mf1)
task_rows.append({"task":"language","accuracy":lang_acc,"macro_f1":lang_mf1,"threshold":None})
lpr,lrc,lf1,lsup=precision_recall_fscore_support(lang_gold,lang_pred,labels=[0,1,2],zero_division=0)
language_rows=[
    {"class":LANGUAGE_LABELS[j],"support":int(lsup[j]),"precision":float(lpr[j]),"recall":float(lrc[j]),"f1":float(lf1[j])}
    for j in range(3)
]

core5_acc=float(np.mean(task_acc))
core5_mf1=float(np.mean(task_mf1))
gated_full6_acc=float((sum(task_acc)+gated_target["exact_set_accuracy"])/6.0)
gated_full6_mf1=float((sum(task_mf1)+gated_target["macro_f1"])/6.0)
facet_full6_acc=float((sum(task_acc)+facet_target["exact_set_accuracy"])/6.0)
facet_full6_mf1=float((sum(task_mf1)+facet_target["macro_f1"])/6.0)

target_delta_pp=100.0*(facet_target["macro_f1"]-gated_target["macro_f1"])
map_delta=facet_target["map"]-gated_target["map"]

result={
    "schema":"M3Offend_Pangea7BHF_FacetGate_SingleSeed_FinalLockedTest_v1",
    "status":"COMPLETE",
    "build_id":LOCKED_TEST_BUILD_ID,
    "protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256,
    "model":"Pangea-7B-HF",
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "seed":RUN_SEED,
    "ordered_test_identity_sha256":TEST_IDENTITY_SHA,
    "rows":932,
    "parent_gated_selected_step":MODEL_CFG["selected_gated_step"],
    "parent_checkpoint_fingerprint":PARENT_CHECKPOINT_FINGERPRINT,
    "selected_facetgate_step":EXPECTED_FACETGATE_STEP,
    "facetgate_checkpoint_sha256":FACET_SELECTED_CHECKPOINT_SHA256,
    "facetgate_validation_threshold_sha256":FACET_VALIDATION_THRESHOLD_SHA256,
    "core5":{"accuracy":core5_acc,"macro_f1":core5_mf1,"tasks":task_rows},
    "gated_target":{k:v for k,v in gated_target.items() if k not in {"rows","pred"}},
    "facetgate_target":{k:v for k,v in facet_target.items() if k not in {"rows","pred"}},
    "delta":{
        "target_macro_f1":float(facet_target["macro_f1"]-gated_target["macro_f1"]),
        "target_macro_f1_pp":target_delta_pp,
        "target_map":map_delta,
    },
    "full6":{
        "gated_accuracy":gated_full6_acc,
        "gated_macro_f1":gated_full6_mf1,
        "facetgate_accuracy":facet_full6_acc,
        "facetgate_macro_f1":facet_full6_mf1,
    },
    "mechanism":{
        "selector_mean":float(selector.mean()),
        "selector_sd":float(selector.std(ddof=1)),
        "selector_min":float(selector.min()),
        "selector_max":float(selector.max()),
        "view_weight_mean":dict(zip(["visual","text","product","difference"],[float(x) for x in view_weights.mean(axis=0)])),
        "image_text_disagreement_mean":float(image_text_disagreement.mean()),
    },
    "locked_test_governance":{
        "checkpoint_selection_from_test":False,
        "threshold_selection_from_test":False,
        "numerical_repair_selection_from_test":False,
        "rerun_based_on_test_result":False,
        "model_test_rows_forwarded":932,
    },
}

# Paper-ready tables
pd.DataFrame(binary_rows).to_csv(OUTPUT_ROOT/"binary_classwise.csv",index=False)
pd.DataFrame(language_rows).to_csv(OUTPUT_ROOT/"language_classwise.csv",index=False)
gt=pd.DataFrame(gated_target["rows"]); gt.insert(0,"branch","Gated")
ft=pd.DataFrame(facet_target["rows"]); ft.insert(0,"branch","FacetGate")
pd.concat([gt,ft],ignore_index=True).to_csv(OUTPUT_ROOT/"target_labelwise.csv",index=False)
pd.DataFrame(task_rows).to_csv(OUTPUT_ROOT/"core5_taskwise.csv",index=False)

manual_row=pd.DataFrame([{
    "seed":RUN_SEED,
    "core5_accuracy":core5_acc,
    "core5_macro_f1":core5_mf1,
    "gated_target_exact_set_accuracy":gated_target["exact_set_accuracy"],
    "gated_target_macro_f1":gated_target["macro_f1"],
    "gated_target_map":gated_target["map"],
    "facetgate_target_exact_set_accuracy":facet_target["exact_set_accuracy"],
    "facetgate_target_macro_f1":facet_target["macro_f1"],
    "facetgate_target_map":facet_target["map"],
    "target_macro_f1_delta_pp":target_delta_pp,
    "gated_full6_accuracy":gated_full6_acc,
    "gated_full6_macro_f1":gated_full6_mf1,
    "facetgate_full6_accuracy":facet_full6_acc,
    "facetgate_full6_macro_f1":facet_full6_mf1,
}])
manual_row.to_csv(OUTPUT_ROOT/"MANUAL_FIVE_SEED_AGGREGATION_ROW.csv",index=False)

pred_df=pd.DataFrame({"official_row_index":td.official_row_index.astype(int)})
for k,t in enumerate(BINARY_TASKS):
    pred_df[f"gold_{t}"]=bin_gold[:,k]
    pred_df[f"p_{t}"]=bin_prob[:,k]
    pred_df[f"pred_{t}"]=bin_preds[t]
pred_df["gold_language"]=[LANGUAGE_LABELS[x] for x in lang_gold]
pred_df["pred_language"]=[LANGUAGE_LABELS[x] for x in lang_pred]
for j,lab in enumerate(TARGET_LABELS):
    pred_df[f"gold_target_{lab}"]=target_gold[:,j]
    pred_df[f"p_gated_target_{lab}"]=gated_target_prob[:,j]
    pred_df[f"pred_gated_target_{lab}"]=gated_target["pred"][:,j]
    pred_df[f"p_facetgate_target_{lab}"]=facet_prob[:,j]
    pred_df[f"pred_facetgate_target_{lab}"]=facet_target["pred"][:,j]
pred_df.to_csv(OUTPUT_ROOT/"locked_test_predictions.csv",index=False)

result_path=OUTPUT_ROOT/f"Pangea_FacetGate_Seed{RUN_SEED}_LOCKED_TEST_RESULT.json"
result_path.write_text(json.dumps(result,indent=2),encoding="utf-8")

manifest={
    "schema":"M3Offend_Pangea7BHF_FacetGate_SingleSeed_LockedTest_Manifest_v1",
    "build_id":LOCKED_TEST_BUILD_ID,
    "seed":RUN_SEED,
    "protocol_sha256":SHARED_FORWARD_PROTOCOL_SHA256,
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "parent_checkpoint_fingerprint":PARENT_CHECKPOINT_FINGERPRINT,
    "facetgate_checkpoint_sha256":FACET_SELECTED_CHECKPOINT_SHA256,
    "facetgate_validation_threshold_sha256":FACET_VALIDATION_THRESHOLD_SHA256,
    "ordered_test_identity_sha256":TEST_IDENTITY_SHA,
    "worker_sha256":WORKER_SHA256,
    "files":{},
}
for p in sorted(OUTPUT_ROOT.glob("*")):
    if p.is_file() and p.suffix.lower() in {".json",".csv",".py"}:
        manifest["files"][p.name]=hashlib.sha256(p.read_bytes()).hexdigest()
(OUTPUT_ROOT/"LOCKED_TEST_MANIFEST.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")

zip_path=Path("/kaggle/working")/f"M3Offend_9318_Pangea7BHF_FacetGate_Seed{RUN_SEED}_FINAL_LOCKED_TEST.zip"
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_ROOT.rglob("*"):
        if p.is_file():
            z.write(p,arcname=str(p.relative_to(OUTPUT_ROOT.parent)))
zip_sha=hashlib.sha256(zip_path.read_bytes()).hexdigest()

print("\n=== PANGEA FACETGATE SINGLE-SEED FINAL LOCKED TEST ===")
print(f"Seed                  : {RUN_SEED}")
print(f"Core-5 Accuracy       : {100*core5_acc:.3f}%")
print(f"Core-5 Macro-F1       : {100*core5_mf1:.3f}%")
print(f"Gated Target Macro-F1 : {100*gated_target['macro_f1']:.3f}%")
print(f"Facet Target Macro-F1 : {100*facet_target['macro_f1']:.3f}%")
print(f"Target MF1 delta      : {target_delta_pp:+.3f} pp")
print(f"Gated Target mAP      : {100*gated_target['map']:.3f}%")
print(f"Facet Target mAP      : {100*facet_target['map']:.3f}%")
print(f"Facet Full-6 Accuracy : {100*facet_full6_acc:.3f}%")
print(f"Facet Full-6 Macro-F1 : {100*facet_full6_mf1:.3f}%")
print("MANUAL MEAN ROW       :",OUTPUT_ROOT/"MANUAL_FIVE_SEED_AGGREGATION_ROW.csv")
print("FINAL ZIP             :",zip_path)
print("FINAL ZIP SHA256      :",zip_sha)
